# LeetCode 428 — Serialize and Deserialize N-ary Tree bằng Python (Hard, Premium)

> **Lưu ý:** đây là bài **Premium** (bị khóa trên LeetCode, cần tài khoản Premium mới xem đề và nộp bài). Phát biểu đề, ví dụ, constraints và khung code dưới đây được viết lại theo bản LeetCode mà mình nắm được. Chỗ nào mình không chắc 100% sẽ ghi rõ là **giả định**.

Đây là một bài **thiết kế class** (design): LeetCode không cho hàm `Solution`, mà cho class `Codec` với hai method. Khung code Python3:

```python
"""
# Definition for a Node.
class Node(object):
    def __init__(self, val: Optional[int] = None, children: Optional[List['Node']] = None):
        if children is None:
            children = []
        self.val = val
        self.children = children
"""

class Codec:
    def serialize(self, root: 'Node') -> str:
        """Encodes a tree to a single string.

        :type root: Node
        :rtype: str
        """

    def deserialize(self, data: str) -> 'Node':
        """Decodes your encoded data to tree.

        :type data: str
        :rtype: Node
        """

# Your Codec object will be instantiated and called as such:
# codec = Codec()
# codec.deserialize(codec.serialize(root))
```

*(Giả định nhỏ: ở một số bài N-ary khác, định nghĩa `Node` không có hai dòng `if children is None`, nên `children` của lá có thể là `None` thay vì `[]`. Code trong notebook luôn viết phòng thủ `node.children or []` để chạy đúng cả hai kiểu.)*

**Mục tiêu của bài học:**

1. Hiểu **cây N-ary** (cây mà mỗi node có thể có **bao nhiêu con cũng được**) từ con số 0: class `Node`, danh sách `children`, root, lá, chiều cao; tự dựng cây từ định dạng level-order của LeetCode `[1,null,3,2,4,null,5,6]` và **in cây dạng ASCII** để nhìn thấy nó.
2. Hiểu **serialize / deserialize** là gì và **vì sao cần** — vì sao không thể "lưu thẳng" một object trong RAM.
3. Tự khám phá vì sao "chỉ ghi các giá trị" là **không đủ** (nhiều cây khác nhau cho cùng một chuỗi — có demo đếm số hình dạng), và rút ra mấu chốt: phải ghi thêm **thông tin "nhóm con kết thúc ở đâu"**.
4. Nắm các công cụ Python cần cho bài: đệ quy và giới hạn đệ quy, `deque`, `str.join` / `split`, cặp **`iter` + `next`**, và stack tự quản lý.
5. Đi qua 4 cách theo thứ tự tự nhiên: **(1) cách "lười" dùng JSON** → **(2) DFS pre-order kèm số con `val,count,...`** (cách chính, trace kỹ) → **(3) DFS với dấu kết thúc `#`** và biến thể **dấu ngoặc** như gợi ý trong đề → **(4) BFS level-order có `null` ngăn nhóm con** (đúng định dạng LeetCode).
6. Chứng minh vì sao đúng, so sánh **độ dài chuỗi**, test **round-trip** `deserialize(serialize(t))` trên cây ngẫu nhiên có `seed` (cả cây rất sâu), tránh các lỗi kinh điển, và nhận dạng pattern cho các bài anh em (297, 431, 449, 652...).

## 1. Đề bài

Đề LeetCode (dịch sát nghĩa):

> **Serialization** (tuần tự hóa) là quá trình chuyển một cấu trúc dữ liệu hay một object thành một dãy bit, để có thể lưu vào file hoặc bộ nhớ đệm, hoặc truyền qua kết nối mạng, rồi sau đó **dựng lại** ở cùng máy tính hoặc ở một máy khác.
>
> Hãy thiết kế thuật toán để serialize và deserialize một **cây N-ary**. Cây N-ary là cây có gốc (rooted tree) trong đó mỗi node có **không quá N con**. Không có giới hạn nào về cách thuật toán của bạn hoạt động. Bạn chỉ cần bảo đảm cây N-ary có thể được serialize thành **một chuỗi** và chuỗi đó có thể được deserialize trở lại **đúng cấu trúc cây ban đầu**.
>
> Ví dụ, bạn có thể serialize cây 3-ary (cây ở Example 2) thành `[1 [3[5 6] 2 4]]`. Đây chỉ là một ví dụ, bạn không bắt buộc theo định dạng này.
>
> Hoặc bạn có thể theo định dạng **level-order** của LeetCode, trong đó **mỗi nhóm con được ngăn cách bằng giá trị `null`**. Ví dụ cây ở Example 1 được serialize thành `[1,null,2,3,4,5,null,null,6,7,null,8,null,9,10,null,null,11,null,12,null,13,null,null,14]`.
>
> Bạn không nhất thiết phải theo các định dạng gợi ý ở trên; có rất nhiều định dạng khác chạy được, hãy sáng tạo và tự nghĩ ra các cách tiếp cận khác nhau.

**Example 1**

```text
Input: root = [1,null,2,3,4,5,null,null,6,7,null,8,null,9,10,null,null,11,null,12,null,13,null,null,14]
Output: [1,null,2,3,4,5,null,null,6,7,null,8,null,9,10,null,null,11,null,12,null,13,null,null,14]
```

**Example 2**

```text
Input: root = [1,null,3,2,4,null,5,6]
Output: [1,null,3,2,4,null,5,6]
```

**Example 3**

```text
Input: root = []
Output: []
```

**Constraints**

- Số node trong cây nằm trong khoảng `[0, 10^4]`.
- `0 <= Node.val <= 10^4`
- Chiều cao của cây N-ary nhỏ hơn hoặc bằng `1000`.
- Không dùng biến thành viên của class / biến global / biến static để lưu trạng thái. Thuật toán encode và decode phải **stateless** (không trạng thái).

### Ký hiệu dùng suốt notebook

- `n` = số node của cây (`0 <= n <= 10^4`, nghĩa là cây **có thể rỗng**).
- `h` = chiều cao của cây = số node trên đường dài nhất từ root xuống một lá (`h <= 1000`).
- `L` = độ dài chuỗi serialize (số ký tự). Vì mỗi giá trị có tối đa 5 chữ số (`10000`), `L` cỡ vài lần `n`.
- `k` = số con của một node cụ thể. Đề **không** cho biết `N` là bao nhiêu, nên lời giải phải chạy với số con tùy ý.

### 1.1 Đề thực sự kiểm tra điều gì?

Đọc kỹ hai dòng comment cuối khung code: LeetCode gọi `codec.serialize(root)` để lấy một chuỗi, đưa chuỗi đó cho `codec.deserialize(...)`, rồi **in cây kết quả theo định dạng level-order** và so với input. Vì thế ở cả 3 ví dụ, Output trùng với Input: "cây bạn dựng lại phải giống hệt cây ban đầu" — cùng giá trị, cùng số con, cùng **thứ tự** các con.

| Chi tiết của đề | Hệ quả khi thiết kế |
|---|---|
| Định dạng chuỗi do ta **tự chọn** | được sáng tạo — miễn là `deserialize(serialize(T))` cho lại đúng cây `T` |
| Phải **stateless** | không được "giấu" cây trong `self` rồi trả lại; mọi thông tin phải nằm **trong chuỗi** |
| Mỗi node có số con **tùy ý** (`N` không biết trước) | khác bài 297 (cây nhị phân luôn có đúng 2 "chỗ" con): ta phải ghi được **một node có bao nhiêu con** |
| Thứ tự con có ý nghĩa | `1 → [3, 2, 4]` khác `1 → [2, 3, 4]`; không được sắp xếp lại con |
| `n` có thể bằng `0` | `serialize(None)` phải ra chuỗi hợp lệ (vd `""`), và `deserialize("")` phải ra `None` |
| `0 <= val <= 10^4` | giá trị **không âm**, có thể nhiều chữ số, có thể **trùng nhau** và có thể **trùng với "số con"** — định dạng không được dựa vào giá trị để đoán vai trò |
| `h <= 1000` | đệ quy sâu tới 1000 tầng — sát giới hạn đệ quy mặc định của Python (`1000`), phải chú ý |

Mỗi dòng của bảng sẽ ứng với một lỗi kinh điển ở mục 15.

## 2. Nền tảng: cây N-ary từ căn bản

### 2.1 Từ vựng và class `Node`

**Cây N-ary** gồm các **node** (nút). Mỗi node có hai thứ:

- `val`: giá trị lưu ở node.
- `children`: một **list** các node con, theo thứ tự từ trái sang phải. Lá có `children = []`.

So với cây nhị phân (mỗi node có `left` và `right`), cây N-ary tổng quát hơn: một node có thể có 0, 1, 2, ... hay 100 con. Cây của Example 2, `[1,null,3,2,4,null,5,6]`, trông như sau:

```text
        1
     /  |  \
    3   2   4
   / \
  5   6
```

| Thuật ngữ | Nghĩa | Ví dụ trên cây Example 2 |
|---|---|---|
| **root** (gốc) | node trên cùng, không có cha | `1` |
| **child** (con) / **parent** (cha) | node ngay dưới / ngay trên | `3`, `2`, `4` là con của `1`; `1` là cha của `3` |
| **leaf** (lá) | node không có con (`children` rỗng) | `5`, `6`, `2`, `4` |
| **subtree** (cây con) | một node cùng toàn bộ con cháu | cây con gốc `3` gồm `{3, 5, 6}` |
| **height** (chiều cao) | số node trên đường dài nhất từ root xuống lá | `3` (đường `1 → 3 → 5`) |
| **degree** (bậc) | số con của một node | bậc của `1` là 3, của `3` là 2, của lá là 0 |

Ta định nghĩa `Node` **y hệt** LeetCode để chạy trong notebook, rồi dựng **bằng tay** cây Example 2. Chú ý cách đọc `root.children[0].children[1]`: "con thứ 0 của root (node `3`), rồi con thứ 1 của nó (node `6`)".

In [1]:
from collections import deque
import json
import random
import sys
import time


class Node(object):
    def __init__(self, val=None, children=None):
        if children is None:
            children = []
        self.val = val
        self.children = children


# Dựng TAY cây Example 2: [1,null,3,2,4,null,5,6]
n5, n6 = Node(5), Node(6)
n3 = Node(3, [n5, n6])
n2, n4 = Node(2), Node(4)
root2 = Node(1, [n3, n2, n4])

print(root2.val)                                  # 1
print([c.val for c in root2.children])            # [3, 2, 4]  (đúng thứ tự)
print(root2.children[0].children[1].val)          # 6
print(len(n2.children), n2.children)              # 0 []  -> node 2 là lá
print(root2)                                      # chỉ là "địa chỉ" object trong RAM, vô nghĩa ngoài chương trình này

1
[3, 2, 4]
6
0 []


### 2.2 Vì sao `children=None` chứ không phải `children=[]`? (bẫy mutable default argument)

Nhiều bạn sẽ "rút gọn" định nghĩa thành `def __init__(self, val=None, children=[])`. Nhìn thì vô hại, nhưng đây là một bẫy kinh điển của Python:

- Giá trị mặc định của tham số được tạo **một lần duy nhất** lúc định nghĩa hàm, không phải mỗi lần gọi.
- Nên mọi node tạo mà không truyền `children` sẽ **dùng chung một list**. Thêm con cho node này = thêm con cho tất cả!

Với bài serialize, bẫy này cực nguy hiểm: hàm deserialize thường tạo `Node(val)` rồi `append` con — nếu dùng bản lỗi, cây dựng ra sẽ "rối" thành một mạng chằng chịt. LeetCode tránh bẫy bằng cách dùng `None` làm mặc định rồi tạo list mới bên trong. Trong notebook, để chắc chắn, các hàm deserialize luôn gọi `Node(val, [])` với một list mới tinh.

In [2]:
class BadNode:
    def __init__(self, val=None, children=[]):     # SAI: list mặc định dùng chung
        self.val = val
        self.children = children


a = BadNode(1)
b = BadNode(2)
a.children.append(BadNode(99))
print("con của b:", [c.val for c in b.children])     # [99]  <- b "tự nhiên" có con!
print("a.children is b.children:", a.children is b.children)   # True

x = Node(1)
y = Node(2)
x.children.append(Node(99))
print("Node đúng, con của y:", [c.val for c in y.children])    # []

con của b: [99]
a.children is b.children: True
Node đúng, con của y: []


### 2.3 Serialize / deserialize là gì? Vì sao cần?

Trong RAM, cây là một đám object nằm rải rác, nối với nhau bằng **tham chiếu** (reference): `root2.children` chứa "địa chỉ" của node `3`, `2`, `4`. Dòng cuối của cell 2.1 in ra kiểu `<__main__.Node object at 0x7f...>` — đó là địa chỉ bộ nhớ, **chỉ có nghĩa trong chương trình đang chạy**. Tắt chương trình đi, hay gửi con số đó sang máy khác, thì nó vô nghĩa.

Muốn **lưu cây vào file**, **gửi cây qua mạng**, hay **lưu vào cache**, ta phải biến cây thành một thứ "phẳng" — một chuỗi ký tự:

- **serialize** (tuần tự hóa): cây → chuỗi. Ví dụ cây Example 2 → `"1,3,3,2,5,0,6,0,2,0,4,0"` (định dạng của cách chính, mục 6).
- **deserialize** (giải tuần tự): chuỗi → cây **mới** (object mới, địa chỉ mới) nhưng **cùng cấu trúc và giá trị**.

Hai thao tác là một **cặp ngược nhau**: `deserialize(serialize(T))` phải giống `T`. Cặp này gọi chung là **codec** (coder–decoder), đúng như tên class LeetCode đặt.

Bài toán thật sự của ta là: **thiết kế một định dạng chuỗi** sao cho từ chuỗi đó có thể dựng lại **duy nhất** một cây. Cái khó nằm ở chữ "duy nhất".

### 2.4 Hàng đợi `deque` và duyệt theo tầng (BFS)

Để hiểu định dạng level-order của LeetCode, ta cần **BFS** (Breadth-First Search — duyệt theo chiều rộng): thăm root, rồi toàn bộ tầng 1, rồi tầng 2... Công cụ chuẩn là **hàng đợi** (queue): vào cuối, ra đầu (FIFO — first in, first out).

Trong Python dùng `collections.deque`:

- `q.append(x)`: thêm vào **cuối**, `O(1)`.
- `q.popleft()`: lấy ra ở **đầu**, `O(1)`. (Dùng `list.pop(0)` cũng lấy được phần tử đầu nhưng tốn `O(n)` vì phải dịch cả list — xem lỗi ở mục 15.)

Thuật toán BFS: cho root vào queue; lặp: lấy node đầu queue ra, "thăm" nó, cho **tất cả con** của nó vào cuối queue. Vì con luôn vào sau những node đang chờ, các node được thăm đúng thứ tự từng tầng.

In [3]:
q = deque()
q.append("A"); q.append("B"); q.append("C")
print(q.popleft(), q)          # A deque(['B', 'C'])  -> ra ở đầu, đúng thứ tự vào

def bfs_order(root):
    if root is None:
        return []
    order, q = [], deque([root])
    while q:
        node = q.popleft()
        order.append(node.val)
        for child in node.children or []:
            q.append(child)
    return order

def levels(root):
    """Gom node theo từng tầng: xử lý trọn 'len(q)' node của tầng hiện tại trong một lượt."""
    if root is None:
        return []
    res, q = [], deque([root])
    while q:
        layer = []
        for _ in range(len(q)):
            node = q.popleft()
            layer.append(node.val)
            q.extend(node.children or [])
        res.append(layer)
    return res

print("BFS:", bfs_order(root2))          # [1, 3, 2, 4, 5, 6]
print("Theo tầng:", levels(root2))       # [[1], [3, 2, 4], [5, 6]]

A deque(['B', 'C'])
BFS: [1, 3, 2, 4, 5, 6]
Theo tầng: [[1], [3, 2, 4], [5, 6]]


### 2.5 Định dạng level-order của LeetCode và helper `build_nary` / `to_leetcode_list`

LeetCode viết cây N-ary thành list theo BFS, với quy tắc: **sau root là một `null`**, rồi với **mỗi node theo thứ tự BFS**, ghi **danh sách con** của nó và kết thúc bằng một `null`. Các `null` thừa ở cuối được cắt bỏ.

Đọc `[1,null,3,2,4,null,5,6]` như sau — mỗi "nhóm" là danh sách con của **node kế tiếp trong thứ tự BFS**:

| Đoạn trong list | Thuộc về | Ý nghĩa |
|---|---|---|
| `1` | root | giá trị root |
| `null` | — | kết thúc "nhóm" của root (root không có anh em) |
| `3, 2, 4, null` | con của `1` (node BFS thứ 1) | `1` có 3 con: `3, 2, 4` |
| `5, 6` | con của `3` (node BFS thứ 2) | `3` có 2 con; các `null` phía sau (của `2`, `4`, `5`, `6` đều rỗng) bị cắt |

**`build_nary(values)`** đi đúng theo bảng trên với một queue và một con trỏ `i`:

| Bước | `i` lúc bắt đầu | `parent` lấy ra từ queue | đọc các giá trị tới `null` | queue sau bước |
|---|---|---|---|---|
| khởi tạo | — | tạo root = `1`, bỏ qua `values[1] = null` | — | `[1]` |
| 1 | 2 | `1` | `3, 2, 4` (dừng ở `values[5] = null`) | `[3, 2, 4]` |
| 2 | 6 | `3` | `5, 6` (hết list) | `[2, 4, 5, 6]` |

Hết list thì dừng — các node còn trong queue (`2, 4, 5, 6`) không có con. **`to_leetcode_list(root)`** làm ngược lại: BFS, ghi con của từng node kèm `null`, cuối cùng cắt `null` thừa. Đây chính là cách LeetCode **in Output**, nên ta dùng nó để so kết quả với đề.

In [4]:
def build_nary(values):
    """List kiểu LeetCode (None = null) -> cây N-ary. Trả về root (hoặc None)."""
    if not values:
        return None
    root = Node(values[0], [])
    q = deque([root])
    i = 2                                   # values[1] là null ngay sau root
    while q and i < len(values):
        parent = q.popleft()
        while i < len(values) and values[i] is not None:
            child = Node(values[i], [])
            parent.children.append(child)
            q.append(child)
            i += 1
        i += 1                              # bỏ qua null kết thúc nhóm con của parent
    return root


def to_leetcode_list(root):
    """Cây N-ary -> list kiểu LeetCode (đã cắt None thừa ở cuối)."""
    if root is None:
        return []
    res = [root.val, None]
    q = deque([root])
    while q:
        node = q.popleft()
        for child in node.children or []:
            res.append(child.val)
            q.append(child)
        res.append(None)
    while res and res[-1] is None:
        res.pop()
    return res


null = None   # để chép thẳng input của LeetCode vào Python
EX1 = [1,null,2,3,4,5,null,null,6,7,null,8,null,9,10,null,null,11,null,12,null,13,null,null,14]
EX2 = [1,null,3,2,4,null,5,6]
EX3 = []

root1 = build_nary(EX1)
print("Example 2 dựng tay   :", to_leetcode_list(root2))
for name, vals in [("EX1", EX1), ("EX2", EX2), ("EX3", EX3)]:
    assert to_leetcode_list(build_nary(vals)) == vals, name
print("build_nary và to_leetcode_list khớp cả 3 ví dụ của đề")

Example 2 dựng tay   : [1, None, 3, 2, 4, None, 5, 6]
build_nary và to_leetcode_list khớp cả 3 ví dụ của đề


### 2.6 Helper in cây dạng ASCII để **nhìn thấy** cây

Vẽ cây N-ary theo kiểu "từ trên xuống" rất khó căn lề khi có nhiều con, nên ta dùng kiểu của lệnh `tree` trong terminal: mỗi node một dòng, thụt lề theo độ sâu; `├──` là con chưa phải cuối, `└──` là con cuối cùng, `│` nối các anh em ở tầng trên.

Hàm `print_tree` viết **không đệ quy** (dùng stack, sẽ giải thích kỹ ở mục 11) để in được cả cây sâu; nó chỉ in tối đa `max_lines` dòng cho gọn. Mẹo nhỏ: con được đẩy vào stack theo thứ tự **ngược**, để con đầu tiên được lấy ra (và in) trước.

In [5]:
def tree_lines(root):
    if root is None:
        return ["(cây rỗng)"]
    lines = [str(root.val)]
    stack = []
    kids = root.children or []
    for i in range(len(kids) - 1, -1, -1):
        stack.append((kids[i], "", i == len(kids) - 1))
    while stack:
        node, prefix, is_last = stack.pop()
        lines.append(prefix + ("└── " if is_last else "├── ") + str(node.val))
        child_prefix = prefix + ("    " if is_last else "│   ")
        kids = node.children or []
        for i in range(len(kids) - 1, -1, -1):
            stack.append((kids[i], child_prefix, i == len(kids) - 1))
    return lines


def print_tree(root, max_lines=30):
    lines = tree_lines(root)
    for line in lines[:max_lines]:
        print(line)
    if len(lines) > max_lines:
        print(f"... (còn {len(lines) - max_lines} dòng nữa)")


print("Example 2:"); print_tree(root2)
print("\nExample 1:"); print_tree(root1)
print("\nExample 3:"); print_tree(build_nary(EX3))

Example 2:
1
├── 3
│   ├── 5
│   └── 6
├── 2
└── 4

Example 1:
1
├── 2
├── 3
│   ├── 6
│   └── 7
│       └── 11
│           └── 14
├── 4
│   └── 8
│       └── 12
└── 5
    ├── 9
    │   └── 13
    └── 10

Example 3:
(cây rỗng)


## 3. Nền tảng: đệ quy trên cây N-ary

### 3.1 Đệ quy (recursion) — "giao việc cho cây con"

**Đệ quy** là khi một hàm gọi lại chính nó trên một bài toán **nhỏ hơn**. Cây là cấu trúc đệ quy tự nhiên: một cây = một root + **danh sách các cây con**. Một hàm đệ quy trên cây luôn có hai phần:

1. **Trường hợp cơ sở** (base case): cây rỗng (`None`) hoặc lá — trả lời ngay.
2. **Bước đệ quy**: gọi hàm cho **từng con**, rồi gộp kết quả.

Ví dụ đếm số node: `count(node) = 1 + count(con_1) + count(con_2) + ...`. Chiều cao: `height(node) = 1 + max(height(con))` (lá có chiều cao 1).

Hai thứ tự duyệt DFS (Depth-First Search — đi sâu trước) quan trọng cho bài này:

- **Pre-order** (tiền thứ tự, LeetCode 589): ghi **node trước**, rồi lần lượt duyệt từng con. Example 2 → `1, 3, 5, 6, 2, 4`.
- **Post-order** (hậu thứ tự, LeetCode 590): duyệt hết các con **trước**, ghi node **sau cùng**. Example 2 → `5, 6, 3, 2, 4, 1`.

Pre-order đặc biệt hợp với serialize: khi đọc lại chuỗi, ta gặp **cha trước con**, nên có thể tạo node cha ngay rồi gắn con vào sau.

In [6]:
def count_nodes(node):
    if node is None:
        return 0
    return 1 + sum(count_nodes(c) for c in node.children or [])

def height(node):
    if node is None:
        return 0
    return 1 + max((height(c) for c in node.children or []), default=0)

def preorder(node, out):
    if node is None:
        return out
    out.append(node.val)              # ghi node TRƯỚC
    for c in node.children or []:
        preorder(c, out)
    return out

def postorder(node, out):
    if node is None:
        return out
    for c in node.children or []:
        postorder(c, out)
    out.append(node.val)              # ghi node SAU CÙNG
    return out

print("Example 2: n =", count_nodes(root2), " h =", height(root2))
print("  pre-order :", preorder(root2, []))
print("  post-order:", postorder(root2, []))
print("Example 1: n =", count_nodes(root1), " h =", height(root1))
print("  pre-order :", preorder(root1, []))

Example 2: n = 6  h = 3
  pre-order : [1, 3, 5, 6, 2, 4]
  post-order: [5, 6, 3, 2, 4, 1]
Example 1: n = 14  h = 5
  pre-order : [1, 2, 3, 6, 7, 11, 14, 4, 8, 12, 5, 9, 13, 10]


### 3.2 Call stack và giới hạn đệ quy — vì sao `h <= 1000` đáng sợ

Mỗi lần một hàm gọi một hàm khác (kể cả gọi chính nó), Python đặt một **frame** (khung lưu biến cục bộ, vị trí đang chạy) lên **call stack**. Hàm đệ quy trên cây đi xuống sâu `h` tầng thì call stack cao khoảng `h` frame.

Để chống tràn bộ nhớ, Python đặt **giới hạn đệ quy** (xem bằng `sys.getrecursionlimit()`, mặc định thường là `1000`). Vượt quá thì ném `RecursionError`. Vấn đề:

- Đề cho `h` tới **1000** — đúng bằng giới hạn mặc định, chưa kể vài chục frame mà chính LeetCode / Jupyter đã dùng sẵn bên dưới.
- Vậy một hàm đệ quy "ngây thơ" **có thể** nổ trên cây lệch sâu 1000 tầng.

Hai cách xử lý (dùng cả hai trong notebook):

1. **Nâng giới hạn**: `sys.setrecursionlimit(10**4)` — đơn giản, đủ cho `h <= 1000`.
2. **Bỏ đệ quy**: tự quản lý một stack bằng list (mục 11) — an toàn với cây sâu bao nhiêu cũng được.

Cell dưới tạo một "cây xích" (mỗi node có đúng 1 con) sâu 1000 tầng, **tạm** đặt giới hạn về `1000` để thấy lỗi, rồi khôi phục.

In [7]:
def make_chain(n, start=0):
    """Cây xích n node: start -> start+1 -> ... (mỗi node 1 con). Dựng bằng vòng lặp, không đệ quy."""
    if n == 0:
        return None
    root = Node(start, [])
    cur = root
    for v in range(start + 1, start + n):
        nxt = Node(v % 10001, [])
        cur.children.append(nxt)
        cur = nxt
    return root

def height_iter(root):
    """Chiều cao bằng BFS theo tầng: không đệ quy."""
    return len(levels(root))

print("giới hạn đệ quy hiện tại:", sys.getrecursionlimit())
chain1000 = make_chain(1000)
print("height_iter(chain1000) =", height_iter(chain1000))

old_limit = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    height(chain1000)
    print("đệ quy chạy được (máy này còn dư frame)")
except RecursionError as e:
    print("height đệ quy với giới hạn 1000 ->", type(e).__name__)
finally:
    sys.setrecursionlimit(old_limit)

sys.setrecursionlimit(10**4)
print("sau khi nâng giới hạn lên 10^4: height(chain1000) =", height(chain1000))

giới hạn đệ quy hiện tại: 1000
height_iter(chain1000) = 1000
height đệ quy với giới hạn 1000 -> RecursionError
sau khi nâng giới hạn lên 10^4: height(chain1000) = 1000


### 3.3 So sánh cấu trúc hai cây: `same_tree`

Để test round-trip, ta cần kiểm tra "cây dựng lại **giống hệt** cây gốc". Không thể dùng `==` (với object tự định nghĩa, `==` chỉ so địa chỉ — hai object khác nhau luôn `False`). Ta tự viết `same_tree(a, b)`: hai cây giống nhau khi

1. cả hai cùng rỗng; hoặc
2. cùng `val`, cùng **số con**, và từng cặp con tương ứng (con thứ `i` với con thứ `i`) cũng giống nhau.

Viết bằng stack (không đệ quy) để so được cả cây sâu `10^4` tầng ở mục 11–12. Ý tưởng: stack chứa các **cặp** node cần so; lấy một cặp ra, so `val` và số con, rồi đẩy các cặp con vào.

In [8]:
def same_tree(a, b):
    stack = [(a, b)]
    while stack:
        x, y = stack.pop()
        if x is None or y is None:
            if x is not y:
                return False
            continue
        if x.val != y.val:
            return False
        cx, cy = x.children or [], y.children or []
        if len(cx) != len(cy):
            return False
        stack.extend(zip(cx, cy))
    return True

print(same_tree(root2, build_nary(EX2)))                   # True: object khác nhau, cấu trúc như nhau
print(root2 == build_nary(EX2))                            # False: '==' chỉ so địa chỉ
print(same_tree(root2, build_nary([1, None, 2, 3, 4, None, 5, 6])))   # False: đổi thứ tự con
print(same_tree(None, None), same_tree(root2, None))       # True False

True
False
False
True False


### 3.4 Công cụ chuỗi: `join`, `split`, `int` — và cái bẫy `"".split(",")`

Ta sẽ biến cây thành một **dãy token** (mỗi token là một mẩu nhỏ: một giá trị, một số đếm, một ký hiệu), rồi nối token bằng dấu phẩy:

- `",".join(["1", "3", "3"])` → `"1,3,3"`: nối list **chuỗi** bằng dấu phẩy. (Phải là chuỗi: `join` một list số nguyên sẽ báo `TypeError`, nên ta `str(...)` trước.)
- `"1,3,3".split(",")` → `["1", "3", "3"]`: tách ngược lại.
- `int("10000")` → `10000`: đổi token về số.

**Vì sao cần dấu phân cách?** Vì giá trị có nhiều chữ số: nối thẳng `"1" + "12"` = `"112"` thì không biết đó là `1, 12` hay `11, 2` hay `112`.

**Bẫy:** `"".split(",")` **không** ra list rỗng mà ra `[""]` (một chuỗi rỗng). Nếu cây rỗng serialize thành `""` thì deserialize phải kiểm tra `if not data: return None` **trước** khi `split`, nếu không `int("")` sẽ nổ `ValueError`.

### 3.5 `iter` và `next`: "băng chuyền token"

Khi deserialize, ta cần đọc token **lần lượt từ trái sang phải**, mỗi token đúng một lần, và nhiều lời gọi đệ quy phải **chia sẻ chung** vị trí đang đọc. Có 3 cách:

| Cách | Viết | Nhược điểm |
|---|---|---|
| `tokens.pop(0)` | dễ nghĩ nhất | mỗi lần `O(n)` vì dịch cả list → tổng `O(n²)` |
| biến chỉ số `i` dùng chung | `i = [0]` hoặc `nonlocal i` | phải nhớ tăng `i` đúng chỗ, dễ sai |
| **iterator** | `it = iter(tokens)`, mỗi lần `next(it)` | gọn, `O(1)` mỗi lần, tự "nhớ" vị trí |

`iter(list)` tạo một **iterator** — hãy tưởng tượng một **băng chuyền**: mỗi lần gọi `next(it)`, băng chuyền nhả ra **token kế tiếp** và nhớ rằng token đó đã được lấy. Mọi hàm cầm cùng một `it` đều thấy cùng một băng chuyền. Hết token mà vẫn gọi `next` thì ném `StopIteration`. Một vòng `for tok in it:` cũng lấy token từ **chính** băng chuyền đó — có thể dừng giữa chừng bằng `return`/`break` và một chỗ khác đọc tiếp từ vị trí ấy (cách 3 sẽ tận dụng điều này).

In [9]:
tokens = ["1", "3", "3", "2"]
s = ",".join(tokens)
print(repr(s), s.split(","))              # '1,3,3,2' ['1', '3', '3', '2']
print("".split(","))                      # ['']  <- KHÔNG phải [] !
try:
    int("")
except ValueError as e:
    print("int('') ->", type(e).__name__)

it = iter(["1", "3", "3", "2", "5"])
print(next(it), next(it))                 # 1 3   -> băng chuyền nhả từng token
def read_pair(it):                        # một hàm khác dùng CHUNG băng chuyền
    return int(next(it)), int(next(it))
print(read_pair(it))                      # (3, 2): tiếp tục đúng từ chỗ dừng
for tok in it:                            # vòng for cũng lấy tiếp từ băng chuyền
    print("còn lại:", tok)                # 5
try:
    next(it)
except StopIteration:
    print("hết token -> StopIteration")

'1,3,3,2' ['1', '3', '3', '2']
['']
int('') -> ValueError
1 3
(3, 2)
còn lại: 5
hết token -> StopIteration


## 4. Cách 1 — Cách "lười": để thư viện JSON lo phần chuỗi

### 4.1 Ý tưởng

Người mới thường nghĩ ngay: "Python có sẵn `json.dumps` biến list/dict lồng nhau thành chuỗi và `json.loads` đọc ngược lại. Vậy chỉ cần đổi cây thành **list lồng nhau** là xong!"

Quy ước: một node ↦ `[val, [danh sách con đã đổi]]`. Lá ↦ `[val, []]`. Cây rỗng ↦ `null` của JSON (`None` của Python).

Example 2 ↦ `[1, [[3, [[5, []], [6, []]]], [2, []], [4, []]]]`.

- `tree_to_obj`: đệ quy, đổi từng con rồi gom vào list.
- `obj_to_tree`: đệ quy ngược lại.
- `separators=(",", ":")` bỏ dấu cách mà `json.dumps` tự thêm, cho chuỗi ngắn hơn.

Cách này **đúng** (JSON giữ nguyên cấu trúc lồng nhau và thứ tự list), nên ta dùng nó làm **baseline** để đối chiếu các cách tự thiết kế phía sau.

In [10]:
def tree_to_obj(node):
    return [node.val, [tree_to_obj(c) for c in node.children or []]]

def obj_to_tree(obj):
    val, kids = obj
    return Node(val, [obj_to_tree(k) for k in kids])

def json_serialize(root):
    return json.dumps(None if root is None else tree_to_obj(root), separators=(",", ":"))

def json_deserialize(data):
    obj = json.loads(data)
    return None if obj is None else obj_to_tree(obj)

s2 = json_serialize(root2)
print(s2)
print_tree(json_deserialize(s2))
for vals in [EX1, EX2, EX3]:
    t = build_nary(vals)
    assert same_tree(json_deserialize(json_serialize(t)), t)
print("JSON round-trip đúng cho cả 3 ví dụ; cây rỗng ->", json_serialize(None))

[1,[[3,[[5,[]],[6,[]]]],[2,[]],[4,[]]]]
1
├── 3
│   ├── 5
│   └── 6
├── 2
└── 4
JSON round-trip đúng cho cả 3 ví dụ; cây rỗng -> null


### 4.2 Cách lười vướng ở đâu?

Cách JSON chạy đúng, vậy sao không dừng ở đây?

1. **Chuỗi dài:** mỗi node tốn thêm `[`, `,`, `[`, `]`, `]` và mỗi lá có thêm một cặp `[]` rỗng. Còn nếu dùng dict `{"val": ..., "children": [...]}` cho "dễ đọc" thì mỗi node còn gánh thêm cả chữ `"val"` và `"children"` — dài gấp mấy lần.
2. **Không học được gì:** ý chính của bài là **tự thiết kế** một định dạng có thể đọc lại duy nhất. Trong phỏng vấn, "dùng JSON" gần như chắc chắn bị hỏi tiếp "không dùng thư viện thì sao?".
3. **Vẫn đệ quy:** `tree_to_obj` / `obj_to_tree` đệ quy sâu `h` tầng, và chính `json.loads` cũng có giới hạn độ lồng nhau (cây xích quá sâu sẽ làm nó ném `RecursionError`).
4. **Che giấu cơ chế:** JSON "hiểu" dấu ngoặc lồng nhau giùm ta. Ở mục 9.3 ta sẽ tự làm điều đó bằng một stack — và thấy rằng dấu ngoặc chính là một cách ghi "nhóm con bắt đầu / kết thúc ở đâu".

Cell dưới đo độ dài trên Example 1 và thử `json.loads` với một chuỗi lồng nhau rất sâu (dựng thẳng chuỗi, không qua cây).

In [11]:
def tree_to_dict(node):
    return {"val": node.val, "children": [tree_to_dict(c) for c in node.children or []]}

print("Example 1 có", count_nodes(root1), "node")
print("  JSON list lồng nhau:", len(json_serialize(root1)), "ký tự")
print("  JSON dict lồng nhau:", len(json.dumps(tree_to_dict(root1), separators=(",", ":"))), "ký tự")
print("  chỉ riêng các giá trị:", len(",".join(str(v) for v in preorder(root1, []))), "ký tự")

deep = "[0,[" * 50_000 + "]]" * 50_000      # 'cây xích' 50 000 tầng viết thẳng thành JSON
try:
    json.loads(deep)
    print("json.loads đọc được chuỗi sâu 50 000 tầng")
except RecursionError as e:
    print("json.loads chuỗi sâu 50 000 tầng ->", type(e).__name__)

Example 1 có 14 node
  JSON list lồng nhau: 94 ký tự
  JSON dict lồng nhau: 332 ký tự
  chỉ riêng các giá trị: 32 ký tự
json.loads chuỗi sâu 50 000 tầng -> RecursionError


## 5. Tự thiết kế định dạng: thử cách đơn giản nhất và xem nó hỏng thế nào

### 5.1 "Chỉ cần ghi các giá trị theo pre-order là đủ?"

Ý tưởng tự nhiên tiếp theo: bỏ hết ngoặc, chỉ ghi giá trị theo pre-order, cách nhau bằng dấu phẩy. Example 2 → `"1,3,5,6,2,4"`. Gọn quá!

Nhưng hãy thử với hai cây 3 node:

```text
Cây A:  1          Cây B:  1
       / \                 |
      2   3                2
                           |
                           3
```

Cả hai đều cho pre-order `1,2,3`. Đọc chuỗi `"1,2,3"`, ta **không biết** `3` là con của `1` (cây A) hay con của `2` (cây B). Chuỗi này **mơ hồ** (ambiguous): một chuỗi ứng với nhiều cây → không thể deserialize đúng.

In [12]:
def preorder_values_only(root):
    return ",".join(str(v) for v in preorder(root, []))

tree_A = Node(1, [Node(2), Node(3)])
tree_B = Node(1, [Node(2, [Node(3)])])
print("A:", preorder_values_only(tree_A)); print_tree(tree_A)
print("B:", preorder_values_only(tree_B)); print_tree(tree_B)
print("Hai cây khác nhau:", not same_tree(tree_A, tree_B),
      "| cùng một chuỗi:", preorder_values_only(tree_A) == preorder_values_only(tree_B))

A: 1,2,3
1
├── 2
└── 3
B: 1,2,3
1
└── 2
    └── 3
Hai cây khác nhau: True | cùng một chuỗi: True


### 5.2 Tệ hơn nữa: với `n` node có cả "họ" hình dạng cùng chung một chuỗi

Nếu ta đánh số các node `1, 2, ..., n` theo pre-order, thì **mọi** hình dạng cây `n` node đều cho cùng chuỗi `"1,2,...,n"`. Có bao nhiêu hình dạng? Cell dưới liệt kê tất cả bằng đệ quy:

- Một cây `s` node = root + một **rừng** (forest — dãy các cây con theo thứ tự) có `s - 1` node.
- Một rừng `m` node = (cây đầu tiên có `t` node) + (rừng `m - t` node còn lại), với `t = 1..m`.

Số hình dạng tăng rất nhanh: `1, 1, 2, 5, 14, 42, 132, ...` (đây là dãy **Catalan**, số thứ `n - 1`). Với 4 node đã có 5 cây khác nhau mà chỉ ghi giá trị thì không phân biệt được cây nào. Để hiển thị gọn trên một dòng, cell dùng ký hiệu `val(con con ...)`.

In [13]:
def all_trees(s):
    """Mọi hình dạng cây s node, mỗi hình dạng là list các cây con (shape lồng nhau)."""
    return all_forests(s - 1)

def all_forests(m):
    if m == 0:
        return [[]]
    res = []
    for t in range(1, m + 1):
        for first in all_trees(t):
            for rest in all_forests(m - t):
                res.append([first] + rest)
    return res

def shape_to_tree(shape, counter):
    counter[0] += 1                       # đánh số theo pre-order: 1, 2, 3, ...
    node = Node(counter[0], [])
    for sub in shape:
        node.children.append(shape_to_tree(sub, counter))
    return node

def inline(node):
    kids = node.children or []
    return str(node.val) + ("(" + " ".join(inline(c) for c in kids) + ")" if kids else "")

print("số hình dạng với n = 1..7:", [len(all_trees(n)) for n in range(1, 8)])
for shape in all_trees(4):
    t = shape_to_tree(shape, [0])
    print(f"  {inline(t):<12} pre-order chỉ giá trị = {preorder_values_only(t)}")

số hình dạng với n = 1..7: [1, 1, 2, 5, 14, 42, 132]
  1(2 3 4)     pre-order chỉ giá trị = 1,2,3,4
  1(2 3(4))    pre-order chỉ giá trị = 1,2,3,4
  1(2(3) 4)    pre-order chỉ giá trị = 1,2,3,4
  1(2(3 4))    pre-order chỉ giá trị = 1,2,3,4
  1(2(3(4)))   pre-order chỉ giá trị = 1,2,3,4


### 5.3 Mấu chốt: chuỗi giá trị thiếu đúng một thông tin — "nhóm con kết thúc ở đâu?"

Khi đọc pre-order `1, 2, 3`, mỗi lần gặp một node mới ta phải quyết định: "node này là **con** của node vừa đọc, hay là **anh em** của nó (con của tổ tiên nào đó)?". Chỉ có giá trị thì không đủ để quyết định. Ta cần thêm **thông tin cấu trúc** cho mỗi node. Có ba cách tự nhiên để ghi thông tin đó:

| Ý tưởng | Ghi thêm gì | Example 2 trở thành | Cách |
|---|---|---|---|
| **Báo trước** số con | sau mỗi giá trị ghi `k` = số con | `1,3,3,2,5,0,6,0,2,0,4,0` | Cách 2 (chính) |
| **Đánh dấu kết thúc** danh sách con | sau khi ghi hết con của một node, ghi `#` | `1,3,5,#,6,#,#,2,#,4,#,#` | Cách 3 |
| **Dấu ngoặc** mở/đóng nhóm con | `[` trước nhóm con, `]` sau nhóm con | `1[3[5 6] 2 4]` | Cách 3 (biến thể) |
| **Theo tầng**, `null` ngăn nhóm | BFS, mỗi node một nhóm con kết thúc bằng `null` | `1,null,3,2,4,null,5,6` | Cách 4 |

So với bài 297 (cây nhị phân): ở đó mỗi node luôn có đúng **2 chỗ** con, nên chỉ cần ghi `#` cho chỗ trống. Ở cây N-ary, số chỗ con của mỗi node **không cố định**, nên phải ghi rõ **bao nhiêu** con (hoặc **đâu là hết**). Đây là điểm khác biệt cốt lõi của bài 428.

Ta bắt đầu với "báo trước số con" vì nó dễ đọc lại nhất: biết trước `k` thì chỉ việc "dựng `k` cây con liên tiếp".

## 6. Cách 2 (cách chính) — DFS pre-order kèm số con: `val,count,...`

### 6.1 Serialize

Duyệt pre-order; tại mỗi node ghi **hai token**: giá trị `val`, rồi **số con** `len(children)`. Sau đó lần lượt serialize từng con.

Với Example 2:

| Thăm node (pre-order) | Số con | Hai token ghi thêm | Ý nghĩa |
|---|---|---|---|
| `1` | 3 | `1, 3` | node 1, "3 cây con tiếp theo là của tôi" |
| `3` | 2 | `3, 2` | node 3, "2 cây con tiếp theo là của tôi" |
| `5` | 0 | `5, 0` | lá |
| `6` | 0 | `6, 0` | lá |
| `2` | 0 | `2, 0` | lá |
| `4` | 0 | `4, 0` | lá |

Kết quả: `"1,3,3,2,5,0,6,0,2,0,4,0"` — đúng `2n = 12` token. Chú ý `"1,3,3,2"`: token thứ hai `3` là **số con** của `1`, token thứ ba `3` là **giá trị** của node con. Hai số giống nhau nhưng **vai trò** khác nhau, và vai trò được quyết định bởi **vị trí** trong chuỗi chứ không phải bởi giá trị. Cây rỗng serialize thành `""`.

Ta gom token vào list `out` rồi `join` một lần ở cuối (nối chuỗi bằng `+=` lặp lại có thể tốn `O(L²)` trong trường hợp xấu).

In [14]:
def count_serialize(root):
    if root is None:
        return ""
    out = []

    def dfs(node):
        kids = node.children or []
        out.append(str(node.val))         # token 1: giá trị
        out.append(str(len(kids)))        # token 2: số con
        for c in kids:                    # rồi tới các cây con, theo đúng thứ tự
            dfs(c)

    dfs(root)
    return ",".join(out)

print("Example 2:", count_serialize(root2))
print("Example 1:", count_serialize(root1))
print("Cây rỗng :", repr(count_serialize(None)))

Example 2: 1,3,3,2,5,0,6,0,2,0,4,0
Example 1: 1,4,2,0,3,2,6,0,7,1,11,1,14,0,4,1,8,1,12,0,5,2,9,1,13,0,10,0
Cây rỗng : ''


### 6.2 Deserialize: đọc token theo đúng nhịp đã ghi

Hàm `build()` "đọc một cây con từ băng chuyền":

1. `val = next(it)`, `k = next(it)` — hai token đầu luôn là giá trị và số con của root cây con này.
2. Tạo `node = Node(val, [])`.
3. Lặp `k` lần: `node.children.append(build())` — mỗi lời gọi `build()` **tự đọc trọn** một cây con, không hơn không kém, rồi băng chuyền dừng đúng ở đầu cây con kế tiếp.
4. Trả `node`.

Vì sao không cần dấu phân cách **giữa các cây con**? Vì mỗi cây con "tự biết" mình dài bao nhiêu: nó bắt đầu bằng `val, k`, và `k` cho biết phải đọc tiếp đúng `k` cây con nữa — khi `build()` trả về, cây con đã kết thúc. (Ở mục 14 ta chứng minh điều này bằng quy nạp.)

Dùng `iter` + `next` (mục 3.5) để mọi lời gọi `build()` chia sẻ chung vị trí đọc. Nhớ kiểm tra chuỗi rỗng **trước** khi `split`.

In [15]:
def count_deserialize(data):
    if not data:                          # cây rỗng: "" -> None (tránh bẫy "".split)
        return None
    it = iter(data.split(","))

    def build():
        val = int(next(it))               # token 1: giá trị
        k = int(next(it))                 # token 2: số con
        node = Node(val, [])
        for _ in range(k):                # dựng đúng k cây con liên tiếp
            node.children.append(build())
        return node

    return build()

t = count_deserialize("1,3,3,2,5,0,6,0,2,0,4,0")
print_tree(t)
print(to_leetcode_list(t))                # [1, None, 3, 2, 4, None, 5, 6]
for vals in [EX1, EX2, EX3]:
    tree = build_nary(vals)
    assert same_tree(count_deserialize(count_serialize(tree)), tree)
    assert to_leetcode_list(count_deserialize(count_serialize(tree))) == vals
print("Round-trip cách 2 đúng cho cả 3 ví dụ")

1
├── 3
│   ├── 5
│   └── 6
├── 2
└── 4
[1, None, 3, 2, 4, None, 5, 6]
Round-trip cách 2 đúng cho cả 3 ví dụ


## 7. Trace từng bước cách chính

### 7.1 Serialize Example 2

Hàm `trace_count_serialize` in mỗi lần `dfs` thăm một node: độ sâu (thụt lề), hai token được ghi thêm, và `out` sau khi ghi. Đối chiếu với bảng ở 6.1: thứ tự thăm là pre-order `1, 3, 5, 6, 2, 4`, và mỗi node đóng góp đúng 2 token.

In [16]:
def trace_count_serialize(root):
    out = []

    def dfs(node, depth):
        kids = node.children or []
        out.append(str(node.val)); out.append(str(len(kids)))
        print(f"{'    ' * depth}thăm {node.val}: ghi ({node.val}, {len(kids)})  -> out = {','.join(out)}")
        for c in kids:
            dfs(c, depth + 1)

    if root is not None:
        dfs(root, 0)
    return ",".join(out)

res = trace_count_serialize(root2)
print("KẾT QUẢ:", res)

thăm 1: ghi (1, 3)  -> out = 1,3
    thăm 3: ghi (3, 2)  -> out = 1,3,3,2
        thăm 5: ghi (5, 0)  -> out = 1,3,3,2,5,0
        thăm 6: ghi (6, 0)  -> out = 1,3,3,2,5,0,6,0
    thăm 2: ghi (2, 0)  -> out = 1,3,3,2,5,0,6,0,2,0
    thăm 4: ghi (4, 0)  -> out = 1,3,3,2,5,0,6,0,2,0,4,0
KẾT QUẢ: 1,3,3,2,5,0,6,0,2,0,4,0


### 7.2 Deserialize `"1,3,3,2,5,0,6,0,2,0,4,0"`

Đây là phần quan trọng nhất. Đánh số token từ 0:

| Chỉ số | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Token | 1 | 3 | 3 | 2 | 5 | 0 | 6 | 0 | 2 | 0 | 4 | 0 |
| Vai trò | val | **k** | val | **k** | val | k | val | k | val | k | val | k |

Diễn biến (mỗi mức thụt lề là một lời gọi `build()` đang mở trên call stack):

| Lời gọi | Đọc | Quyết định |
|---|---|---|
| `build#1` | `t0=1`, `t1=3` | tạo `Node(1)`, cần 3 con |
| &nbsp;&nbsp;`build#2` | `t2=3`, `t3=2` | tạo `Node(3)`, cần 2 con |
| &nbsp;&nbsp;&nbsp;&nbsp;`build#3` | `t4=5`, `t5=0` | tạo `Node(5)`, 0 con → trả về; gắn làm con 1/2 của `3` |
| &nbsp;&nbsp;&nbsp;&nbsp;`build#4` | `t6=6`, `t7=0` | tạo `Node(6)` → trả về; gắn làm con 2/2 của `3` |
| &nbsp;&nbsp;`build#2` xong | — | `Node(3)` đủ 2 con → trả về; gắn làm con 1/3 của `1` |
| &nbsp;&nbsp;`build#5` | `t8=2`, `t9=0` | `Node(2)` → con 2/3 của `1` |
| &nbsp;&nbsp;`build#6` | `t10=4`, `t11=0` | `Node(4)` → con 3/3 của `1` |
| `build#1` xong | — | trả root; băng chuyền vừa hết token |

Hàm trace dưới dùng một chỉ số `pos` thay cho iterator chỉ để **in ra vị trí token** đang đọc; logic giống hệt `count_deserialize`. Ví dụ thứ hai là một cây "khó" hơn: giá trị `0` (dễ nhầm với "0 con"), giá trị trùng nhau và trùng với số con.

In [17]:
def trace_count_deserialize(data):
    if not data:
        print("chuỗi rỗng -> None"); return None
    toks = data.split(",")
    pos = [0]

    def build(depth):
        ind = "    " * depth
        i = pos[0]
        val, k = int(toks[i]), int(toks[i + 1]); pos[0] += 2
        print(f"{ind}đọc t{i}={val} (val), t{i+1}={k} (số con) -> Node({val}), cần {k} con")
        node = Node(val, [])
        for j in range(k):
            child = build(depth + 1)
            node.children.append(child)
            print(f"{ind}  gắn {child.val} làm con {j + 1}/{k} của {val}")
        return node

    root = build(0)
    print(f"đã đọc {pos[0]}/{len(toks)} token")
    return root

r = trace_count_deserialize("1,3,3,2,5,0,6,0,2,0,4,0")
print("Cây dựng được khớp Example 2:", same_tree(r, root2))
print("-" * 40)
tricky = trace_count_deserialize("2,2,0,1,2,0,2,0")
print_tree(tricky)

đọc t0=1 (val), t1=3 (số con) -> Node(1), cần 3 con
    đọc t2=3 (val), t3=2 (số con) -> Node(3), cần 2 con
        đọc t4=5 (val), t5=0 (số con) -> Node(5), cần 0 con
      gắn 5 làm con 1/2 của 3
        đọc t6=6 (val), t7=0 (số con) -> Node(6), cần 0 con
      gắn 6 làm con 2/2 của 3
  gắn 3 làm con 1/3 của 1
    đọc t8=2 (val), t9=0 (số con) -> Node(2), cần 0 con
  gắn 2 làm con 2/3 của 1
    đọc t10=4 (val), t11=0 (số con) -> Node(4), cần 0 con
  gắn 4 làm con 3/3 của 1
đã đọc 12/12 token
Cây dựng được khớp Example 2: True
----------------------------------------
đọc t0=2 (val), t1=2 (số con) -> Node(2), cần 2 con
    đọc t2=0 (val), t3=1 (số con) -> Node(0), cần 1 con
        đọc t4=2 (val), t5=0 (số con) -> Node(2), cần 0 con
      gắn 2 làm con 1/1 của 0
  gắn 0 làm con 1/2 của 2
    đọc t6=2 (val), t7=0 (số con) -> Node(2), cần 0 con
  gắn 2 làm con 2/2 của 2
đã đọc 8/8 token
2
├── 0
│   └── 2
└── 2


## 8. Code nộp LeetCode (cách chính)

Chuyển hai hàm ở mục 6 vào class `Codec` với đúng tên method và chữ ký của LeetCode. Ba chi tiết cần nhớ:

1. **Stateless:** không lưu gì vào `self`; hàm `build` lồng bên trong `deserialize`, iterator `it` là biến cục bộ. Mỗi lần gọi là độc lập.
2. **`node.children or []`:** phòng trường hợp `children` là `None`.
3. **Giới hạn đệ quy:** `h <= 1000` nên đệ quy sâu tối đa khoảng 1000 tầng; nâng giới hạn lên `10**4` ở đầu file cho chắc. (Nếu muốn an toàn tuyệt đối, xem bản không đệ quy ở mục 11.)

Bên trong class, viết `'Node'` trong chú thích kiểu (type hint) dưới dạng chuỗi là đúng cách LeetCode viết — `Node` được LeetCode định nghĩa sẵn.

In [18]:
import sys
sys.setrecursionlimit(10**4)


class Codec:
    def serialize(self, root: 'Node') -> str:
        """Encodes a tree to a single string."""
        if root is None:
            return ""
        out = []

        def dfs(node):
            kids = node.children or []
            out.append(str(node.val))
            out.append(str(len(kids)))
            for child in kids:
                dfs(child)

        dfs(root)
        return ",".join(out)

    def deserialize(self, data: str) -> 'Node':
        """Decodes your encoded data to tree."""
        if not data:
            return None
        it = iter(data.split(","))

        def build():
            val = int(next(it))
            k = int(next(it))
            node = Node(val, [])
            for _ in range(k):
                node.children.append(build())
            return node

        return build()

### 8.2 Mô phỏng đúng cách LeetCode chấm

Hai helper:

- `run_codec_leetcode(codec_cls, values)`: dựng cây từ list kiểu LeetCode → serialize → deserialize → in lại thành list. Cố ý dùng **hai object khác nhau** cho serialize và deserialize để "bắt quả tang" codec nào lén giữ trạng thái trong `self`.
- `run_commands(commands, args, classes)`: chạy chuỗi lệnh kiểu LeetCode như `["Codec", "serialize", "deserialize"]` với tham số `[[], [root], ["$prev"]]`, trong đó `"$prev"` nghĩa là "kết quả của lệnh ngay trước". Đây là khuôn chung cho mọi bài thiết kế class.

Test gồm 3 ví dụ của đề và các edge case: một node, giá trị `0` và `10000`, cây "sao" (root có 1000 con), cây xích 1000 tầng (đúng giới hạn `h` của đề).

In [19]:
def run_codec_leetcode(codec_cls, values):
    root = build_nary(values)
    ser, deser = codec_cls(), codec_cls()        # hai object khác nhau -> buộc phải stateless
    data = ser.serialize(root)
    assert isinstance(data, str)
    return to_leetcode_list(deser.deserialize(data))


def run_commands(commands, args, classes):
    obj, prev, outputs = None, None, []
    for cmd, arg in zip(commands, args):
        arg = [prev if a == "$prev" else a for a in arg]
        if cmd in classes:
            obj = classes[cmd](*arg); outputs.append(None)
        else:
            prev = getattr(obj, cmd)(*arg); outputs.append(prev)
    return outputs


def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<28} PASS")


check("Example 1", run_codec_leetcode(Codec, EX1), EX1)
check("Example 2", run_codec_leetcode(Codec, EX2), EX2)
check("Example 3 (rỗng)", run_codec_leetcode(Codec, EX3), EX3)
check("một node", run_codec_leetcode(Codec, [7]), [7])
check("giá trị 0 và 10000", run_codec_leetcode(Codec, [0, None, 10000, 0]), [0, None, 10000, 0])
star = [1, None] + list(range(1000))
check("cây sao 1000 con", run_codec_leetcode(Codec, star), star)
chain_list = to_leetcode_list(make_chain(1000))
check("cây xích h = 1000", run_codec_leetcode(Codec, chain_list), chain_list)

outs = run_commands(["Codec", "serialize", "deserialize"], [[], [build_nary(EX2)], ["$prev"]], {"Codec": Codec})
print("run_commands:", outs[:2], "->", to_leetcode_list(outs[2]))

Example 1                    PASS
Example 2                    PASS
Example 3 (rỗng)             PASS
một node                     PASS
giá trị 0 và 10000           PASS
cây sao 1000 con             PASS
cây xích h = 1000            PASS
run_commands: [None, '1,3,3,2,5,0,6,0,2,0,4,0'] -> [1, None, 3, 2, 4, None, 5, 6]


## 9. Cách 3 — DFS với dấu kết thúc danh sách con `#`

### 9.1 Ý tưởng

Thay vì **báo trước** "tôi có `k` con", ta **báo sau**: ghi giá trị node, ghi lần lượt các cây con, rồi ghi một **sentinel** (lính canh — ký hiệu đặc biệt không thể là giá trị) `#` nghĩa là "danh sách con của node này **đến đây là hết**". Vì `val` luôn là số không âm, `#` không bao giờ trùng với một giá trị.

Example 2 → `1,3,5,#,6,#,#,2,#,4,#,#`. Đọc từng phần:

```text
1  3  5  #  6  #  #  2  #  4  #  #
│  │  └──┘  └──┘  │  └──┘  └──┘  │
│  │  lá 5  lá 6  │  lá 2  lá 4  │
│  └──────────────┘ hết con của 3│
└────────────────────────────────┘ hết con của 1
```

Mỗi node đóng góp đúng **2 token** (giá trị + một `#` của chính nó), nên cũng `2n` token như cách 2. Mỗi `#` giống như một dấu "đóng ngoặc", còn giá trị là "mở ngoặc".

**Deserialize** rất đẹp nhờ iterator: `build(tok)` nhận token giá trị đầu tiên, tạo node, rồi `for tok in it:` — gặp `#` thì "hết con, trả về"; gặp giá trị thì đó là con mới, gọi `build(tok)` đệ quy. Vòng `for` lồng nhau cùng kéo từ **một** băng chuyền, nên khi lời gọi con trả về, vòng `for` của cha tiếp tục đúng chỗ.

### 9.2 Trace deserialize

`trace_sentinel_deserialize` in mỗi token được đọc và **chuỗi các node đang mở** (từ root tới node hiện tại — chính là những lời gọi `build` còn trên call stack). Để ý: mỗi `#` "đóng" node ở cuối chuỗi đó.

In [20]:
def sentinel_serialize(root):
    if root is None:
        return ""
    out = []

    def dfs(node):
        out.append(str(node.val))
        for c in node.children or []:
            dfs(c)
        out.append("#")                    # hết danh sách con của node này

    dfs(root)
    return ",".join(out)


def sentinel_deserialize(data):
    if not data:
        return None
    it = iter(data.split(","))

    def build(val_tok):
        node = Node(int(val_tok), [])
        for tok in it:                     # kéo tiếp từ CHUNG một băng chuyền
            if tok == "#":
                return node                # hết con
            node.children.append(build(tok))
        return node                        # (chuỗi hợp lệ không tới được dòng này)

    return build(next(it))


def trace_sentinel_deserialize(data):
    it = iter(data.split(","))
    open_path = []

    def build(val_tok):
        node = Node(int(val_tok), [])
        open_path.append(node.val)
        print(f"đọc {val_tok:>2}: mở node {val_tok:<3} đang mở = {open_path}")
        for tok in it:
            if tok == "#":
                open_path.pop()
                print(f"đọc  #: đóng node {node.val:<3} đang mở = {open_path}")
                return node
            node.children.append(build(tok))
        return node

    return build(next(it))


s = sentinel_serialize(root2)
print("Example 2:", s)
r = trace_sentinel_deserialize(s)
print("khớp Example 2:", same_tree(r, root2))
for vals in [EX1, EX2, EX3]:
    tree = build_nary(vals)
    assert same_tree(sentinel_deserialize(sentinel_serialize(tree)), tree)
print("Round-trip cách 3 đúng cho cả 3 ví dụ")

Example 2: 1,3,5,#,6,#,#,2,#,4,#,#
đọc  1: mở node 1   đang mở = [1]
đọc  3: mở node 3   đang mở = [1, 3]
đọc  5: mở node 5   đang mở = [1, 3, 5]
đọc  #: đóng node 5   đang mở = [1, 3]
đọc  6: mở node 6   đang mở = [1, 3, 6]
đọc  #: đóng node 6   đang mở = [1, 3]
đọc  #: đóng node 3   đang mở = [1]
đọc  2: mở node 2   đang mở = [1, 2]
đọc  #: đóng node 2   đang mở = [1]
đọc  4: mở node 4   đang mở = [1, 4]
đọc  #: đóng node 4   đang mở = [1]
đọc  #: đóng node 1   đang mở = []
khớp Example 2: True
Round-trip cách 3 đúng cho cả 3 ví dụ


### 9.3 Biến thể: dấu ngoặc — định dạng gợi ý trong đề `[1 [3[5 6] 2 4]]`

Đề gợi ý một định dạng dùng **dấu ngoặc**. Ta làm một bản gọn: node ↦ `val`, và **nếu có con** thì thêm `[` + các con cách nhau bằng dấu cách + `]`. Example 2 ↦ `1[3[5 6] 2 4]`. Điểm hay: **lá không tốn ký hiệu nào**, chỉ node có con mới tốn một cặp ngoặc — nên chuỗi thường **ngắn hơn** hai cách trước (đo ở mục 13).

Deserialize bằng **stack** tự quản lý, quét từng ký tự từ trái sang phải — không cần đệ quy:

| Gặp | Hành động |
|---|---|
| một dãy chữ số | tạo node mới; gắn vào `children` của node **trên đỉnh stack** (nếu stack rỗng, đó là root); nhớ nó là `last` |
| `[` | node `last` sắp có con → **đẩy** `last` vào stack (nó thành "cha hiện tại") |
| `]` | hết con của cha hiện tại → **bật** khỏi stack |
| dấu cách | chỉ để ngăn cách hai số, bỏ qua |

Đây chính là cách JSON (mục 4) "hiểu" ngoặc lồng nhau — giờ ta tự làm. (Vì đề cho `val >= 0`, chỉ cần nhận chữ số; nếu có số âm phải xử lý thêm dấu `-`.)

In [21]:
def bracket_serialize(root):
    if root is None:
        return ""
    out = []

    def dfs(node):
        out.append(str(node.val))
        kids = node.children or []
        if kids:
            out.append("[")
            for i, c in enumerate(kids):
                if i:
                    out.append(" ")
                dfs(c)
            out.append("]")

    dfs(root)
    return "".join(out)


def bracket_deserialize(data, trace=False):
    if not data:
        return None
    stack, root, last, i = [], None, None, 0
    while i < len(data):
        ch = data[i]
        if ch.isdigit():
            j = i
            while j < len(data) and data[j].isdigit():
                j += 1
            node = Node(int(data[i:j]), [])
            if stack:
                stack[-1].children.append(node)
            else:
                root = node
            last = node
            action, i = f"tạo {node.val}" + (f", gắn vào {stack[-1].val}" if stack else " (root)"), j
        else:
            if ch == "[":
                stack.append(last); action = f"đẩy {last.val}"
            elif ch == "]":
                action = f"bật {stack.pop().val}"
            else:
                action = "bỏ qua dấu cách"
            i += 1
        if trace:
            print(f"{data[:i]:<14} {action:<22} stack = {[x.val for x in stack]}")
    return root


s = bracket_serialize(root2)
print("Example 2:", s, "\n")
r = bracket_deserialize(s, trace=True)
print("\nkhớp Example 2:", same_tree(r, root2))
for vals in [EX1, EX2, EX3]:
    tree = build_nary(vals)
    assert same_tree(bracket_deserialize(bracket_serialize(tree)), tree)
print("Example 1:", bracket_serialize(root1))

Example 2: 1[3[5 6] 2 4] 

1              tạo 1 (root)           stack = []
1[             đẩy 1                  stack = [1]
1[3            tạo 3, gắn vào 1       stack = [1]
1[3[           đẩy 3                  stack = [1, 3]
1[3[5          tạo 5, gắn vào 3       stack = [1, 3]
1[3[5          bỏ qua dấu cách        stack = [1, 3]
1[3[5 6        tạo 6, gắn vào 3       stack = [1, 3]
1[3[5 6]       bật 3                  stack = [1]
1[3[5 6]       bỏ qua dấu cách        stack = [1]
1[3[5 6] 2     tạo 2, gắn vào 1       stack = [1]
1[3[5 6] 2     bỏ qua dấu cách        stack = [1]
1[3[5 6] 2 4   tạo 4, gắn vào 1       stack = [1]
1[3[5 6] 2 4]  bật 1                  stack = []

khớp Example 2: True
Example 1: 1[2 3[6 7[11[14]]] 4[8[12]] 5[9[13] 10]]


## 10. Cách 4 — BFS level-order với `null` ngăn cách nhóm con (định dạng LeetCode)

### 10.1 Serialize

Đây là cách LeetCode tự dùng để in input/output, nên rất đáng biết. Quy tắc (mục 2.5): ghi root, ghi `null`; rồi BFS — mỗi lần lấy một node ra khỏi queue, ghi **toàn bộ con** của nó và một `null` kết thúc nhóm; cuối cùng cắt các `null` thừa.

Trace Example 2:

| Lấy ra khỏi queue | Ghi thêm | queue sau đó |
|---|---|---|
| (khởi tạo) | `1, null` | `[1]` |
| `1` | `3, 2, 4, null` | `[3, 2, 4]` |
| `3` | `5, 6, null` | `[2, 4, 5, 6]` |
| `2` | `null` | `[4, 5, 6]` |
| `4` | `null` | `[5, 6]` |
| `5` | `null` | `[6]` |
| `6` | `null` | `[]` |

Ra `1,null,3,2,4,null,5,6,null,null,null,null` → cắt 4 `null` cuối → `1,null,3,2,4,null,5,6`, đúng Output của LeetCode.

**Vì sao cần `null` ngay sau root?** Để mọi nhóm đều có dạng "các giá trị + `null`": có thể coi root là "nhóm con" của một node ảo. Bỏ `null` này đi thì người đọc sẽ hiểu nhầm `3, 2, 4` là anh em của root (lỗi ở mục 15).

In [22]:
def bfs_serialize(root, trace=False):
    if root is None:
        return ""
    out = [str(root.val), "null"]
    q = deque([root])
    while q:
        node = q.popleft()
        added = []
        for c in node.children or []:
            out.append(str(c.val)); added.append(str(c.val))
            q.append(c)
        out.append("null")
        if trace:
            print(f"lấy {node.val}: ghi {added + ['null']}, queue = {[x.val for x in q]}")
    while out[-1] == "null":                 # cắt null thừa ở cuối
        out.pop()
    return ",".join(out)

s = bfs_serialize(root2, trace=True)
print("Example 2:", s)
print("Example 1:", bfs_serialize(root1))
expected1 = ",".join("null" if v is None else str(v) for v in EX1)
print("Example 1 trùng đúng chuỗi của đề:", bfs_serialize(root1) == expected1)

lấy 1: ghi ['3', '2', '4', 'null'], queue = [3, 2, 4]
lấy 3: ghi ['5', '6', 'null'], queue = [2, 4, 5, 6]
lấy 2: ghi ['null'], queue = [4, 5, 6]
lấy 4: ghi ['null'], queue = [5, 6]
lấy 5: ghi ['null'], queue = [6]
lấy 6: ghi ['null'], queue = []
Example 2: 1,null,3,2,4,null,5,6
Example 1: 1,null,2,3,4,5,null,null,6,7,null,8,null,9,10,null,null,11,null,12,null,13,null,null,14
Example 1 trùng đúng chuỗi của đề: True


### 10.2 Deserialize: queue các "cha đang chờ nhận con"

Ngược lại: tạo root từ token đầu, bỏ qua `null` thứ hai, cho root vào queue. Sau đó đọc tiếp từng **nhóm**: mỗi nhóm (dãy giá trị kết thúc bằng `null`, hoặc kết thúc vì hết chuỗi) là danh sách con của **node kế tiếp trong queue**. Mỗi con mới tạo cũng được cho vào queue, chờ tới lượt nhận nhóm con của nó.

Vì sao "nhóm thứ `g`" luôn thuộc về "node thứ `g` lấy ra khỏi queue"? Vì cả serialize lẫn deserialize đều xử lý node theo **cùng một thứ tự BFS**: serialize ghi nhóm của node theo thứ tự lấy ra khỏi queue, deserialize gán nhóm theo đúng thứ tự đó. Các `null` cuối bị cắt chỉ ứng với những node không có con — khi hết chuỗi, ta dừng và các node còn lại trong queue tự nhiên là lá.

Hàm này thực chất là `build_nary` ở mục 2.5, chỉ khác là đầu vào là chuỗi.

In [23]:
def bfs_deserialize(data, trace=False):
    if not data:
        return None
    toks = data.split(",")
    root = Node(int(toks[0]), [])
    q = deque([root])
    i = 2                                   # toks[1] là "null" sau root
    while q and i < len(toks):
        parent = q.popleft()
        group = []
        while i < len(toks) and toks[i] != "null":
            child = Node(int(toks[i]), [])
            parent.children.append(child); q.append(child); group.append(child.val)
            i += 1
        i += 1                              # bỏ qua "null" kết thúc nhóm
        if trace:
            print(f"cha {parent.val}: nhận nhóm con {group}, queue = {[x.val for x in q]}")
    return root

r = bfs_deserialize("1,null,3,2,4,null,5,6", trace=True)
print_tree(r)
for vals in [EX1, EX2, EX3]:
    tree = build_nary(vals)
    assert same_tree(bfs_deserialize(bfs_serialize(tree)), tree)
print("Round-trip cách 4 đúng cho cả 3 ví dụ")

cha 1: nhận nhóm con [3, 2, 4], queue = [3, 2, 4]
cha 3: nhận nhóm con [5, 6], queue = [2, 4, 5, 6]
1
├── 3
│   ├── 5
│   └── 6
├── 2
└── 4
Round-trip cách 4 đúng cho cả 3 ví dụ


### 10.3 Bản nộp LeetCode của cách BFS

Ưu điểm lớn của BFS: **không đệ quy** chút nào — chỉ dùng queue, nên không lo giới hạn đệ quy dù cây sâu bao nhiêu. Bộ nhớ phụ là kích thước queue, tối đa bằng **độ rộng** lớn nhất của cây (số node trên tầng đông nhất). Đổi lại, code dài hơn và có nhiều chi tiết dễ sai hơn (`null` sau root, nhóm rỗng, cắt `null` cuối).

In [24]:
class CodecBFS:
    def serialize(self, root: 'Node') -> str:
        if root is None:
            return ""
        out = [str(root.val), "null"]
        q = deque([root])
        while q:
            node = q.popleft()
            for child in node.children or []:
                out.append(str(child.val))
                q.append(child)
            out.append("null")
        while out[-1] == "null":
            out.pop()
        return ",".join(out)

    def deserialize(self, data: str) -> 'Node':
        if not data:
            return None
        toks = data.split(",")
        root = Node(int(toks[0]), [])
        q = deque([root])
        i = 2
        while q and i < len(toks):
            parent = q.popleft()
            while i < len(toks) and toks[i] != "null":
                child = Node(int(toks[i]), [])
                parent.children.append(child)
                q.append(child)
                i += 1
            i += 1
        return root


for name, vals in [("Example 1", EX1), ("Example 2", EX2), ("Example 3", EX3), ("cây sao", star), ("cây xích", chain_list)]:
    check("BFS " + name, run_codec_leetcode(CodecBFS, vals), vals)

BFS Example 1                PASS
BFS Example 2                PASS
BFS Example 3                PASS
BFS cây sao                  PASS
BFS cây xích                 PASS


## 11. Cây sâu: bản không đệ quy của cách chính (stack tự quản lý)

Với `h <= 1000` và `sys.setrecursionlimit(10**4)`, bản đệ quy đã đủ cho LeetCode. Nhưng ở ngoài đời (hoặc khi phỏng vấn viên hỏi "nếu cây sâu 1 triệu tầng thì sao?"), ta cần bản **không đệ quy**. Ý tưởng: thay call stack của Python bằng một **list làm stack** do ta tự quản lý.

**Serialize lặp:** stack chứa các node **chờ được thăm**. Lấy node trên đỉnh ra, ghi `val, k`, rồi đẩy các con vào theo thứ tự **ngược** (con cuối vào trước) — để con **đầu tiên** nằm trên đỉnh và được thăm trước. Như vậy thứ tự thăm đúng là pre-order.

**Deserialize lặp:** stack chứa các cặp `[node, số con còn thiếu]` — đúng những gì mỗi lời gọi `build()` "nhớ" trên call stack. Lặp:

- Nếu node trên đỉnh **đã đủ con** (còn thiếu `0`) → bật nó ra (giống `build()` trả về).
- Ngược lại đọc 2 token `val, k` → tạo con mới, gắn vào node trên đỉnh, giảm "còn thiếu" của đỉnh đi 1, rồi đẩy `[con, k]` vào stack (giống gọi `build()` cho con).

Sau đó kiểm tra trên cây xích `10^4` tầng (vượt xa constraint, chỉ để chứng minh độ bền) và đối chiếu chuỗi với bản đệ quy.

In [25]:
def count_serialize_iter(root):
    if root is None:
        return ""
    out, stack = [], [root]
    while stack:
        node = stack.pop()
        kids = node.children or []
        out.append(str(node.val)); out.append(str(len(kids)))
        for c in reversed(kids):            # con đầu tiên nằm trên đỉnh -> thăm trước
            stack.append(c)
    return ",".join(out)


def count_deserialize_iter(data):
    if not data:
        return None
    toks = data.split(",")
    root = Node(int(toks[0]), [])
    stack = [[root, int(toks[1])]]          # [node, số con còn thiếu]
    i = 2
    while stack:
        top = stack[-1]
        if top[1] == 0:                     # đủ con -> "trả về"
            stack.pop()
            continue
        child = Node(int(toks[i]), [])
        k = int(toks[i + 1]); i += 2
        top[0].children.append(child)
        top[1] -= 1
        stack.append([child, k])
    return root


assert count_serialize_iter(root1) == count_serialize(root1)
assert same_tree(count_deserialize_iter(count_serialize(root2)), root2)

deep_chain = make_chain(10_000)
t0 = time.perf_counter()
s_deep = count_serialize_iter(deep_chain)
back = count_deserialize_iter(s_deep)
print(f"cây xích 10^4 tầng: {len(s_deep)} ký tự, round-trip đúng = {same_tree(back, deep_chain)},"
      f" chiều cao dựng lại = {height_iter(back)}, mất {time.perf_counter() - t0:.3f}s")
deep_bracket = "[".join(str(v) for v in range(10_000)) + "]" * 9_999   # "0[1[2[...]]]" viết tay
print("BFS (serialize + deserialize) trên cây xích 10^4:", same_tree(bfs_deserialize(bfs_serialize(deep_chain)), deep_chain))
print("bracket_deserialize (dùng stack) trên chuỗi ngoặc sâu 10^4:", same_tree(bracket_deserialize(deep_bracket), deep_chain))

cây xích 10^4 tầng: 68889 ký tự, round-trip đúng = True, chiều cao dựng lại = 10000, mất 0.025s


BFS (serialize + deserialize) trên cây xích 10^4: True


bracket_deserialize (dùng stack) trên chuỗi ngoặc sâu 10^4: True


## 12. Đối chiếu chéo mọi cách trên cây ngẫu nhiên có `seed`

Test vài ví dụ chưa đủ tin. Ta sinh **cây ngẫu nhiên** bằng `random_nary(n, mode)`: bắt đầu với root, mỗi node mới được gắn làm con (thêm vào **cuối** danh sách con) của một node đã có:

- `mode="wide"`: chọn cha ngẫu nhiên trong **tất cả** node → cây "bè", nhiều con, thấp.
- `mode="deep"`: chọn cha trong **3 node mới nhất** → cây "lệch", rất sâu (chiều cao cỡ một nửa `n`).

`random.seed(...)` cố định để mỗi lần chạy notebook ra đúng cùng một bộ cây (tái lập được). Với mỗi cây, ta kiểm tra:

1. **Round-trip** `same_tree(deserialize(serialize(t)), t)` cho cả 6 codec: JSON, đếm con (đệ quy và lặp), `#`, ngoặc, BFS.
2. Bản đếm con đệ quy và lặp cho **cùng một chuỗi**.
3. Chuỗi BFS khớp `to_leetcode_list` (đúng định dạng LeetCode).
4. Giá trị nhỏ (`0..3`) để cố tình tạo nhiều giá trị trùng nhau và trùng với "số con" — xem định dạng có bị nhầm vai trò không.

In [26]:
def random_nary(n, mode="wide", lo=0, hi=10**4):
    if n == 0:
        return None
    nodes = [Node(random.randint(lo, hi), [])]
    for _ in range(n - 1):
        if mode == "wide":
            parent = random.choice(nodes)
        else:
            parent = nodes[-1 - random.randrange(min(3, len(nodes)))]
        child = Node(random.randint(lo, hi), [])
        parent.children.append(child)
        nodes.append(child)
    return nodes[0]


CODECS = {
    "json":       (json_serialize, json_deserialize),
    "count":      (count_serialize, count_deserialize),
    "count_iter": (count_serialize_iter, count_deserialize_iter),
    "sentinel":   (sentinel_serialize, sentinel_deserialize),
    "bracket":    (bracket_serialize, bracket_deserialize),
    "bfs":        (bfs_serialize, bfs_deserialize),
}

random.seed(428)
n_trees, max_h = 0, 0
for trial in range(300):
    n = random.choice([0, 1, 2, 3, random.randint(4, 30), random.randint(30, 300)])
    mode = random.choice(["wide", "deep"])
    hi = random.choice([3, 10**4])
    t = random_nary(n, mode, 0, hi)
    for name, (ser, de) in CODECS.items():
        assert same_tree(de(ser(t)), t), (name, trial)
    assert count_serialize(t) == count_serialize_iter(t)
    assert bfs_serialize(t) == ",".join("null" if v is None else str(v) for v in to_leetcode_list(t))
    n_trees += 1; max_h = max(max_h, height_iter(t))
print(f"{n_trees} cây ngẫu nhiên x {len(CODECS)} codec: round-trip đều đúng (cây cao nhất h = {max_h})")

big_deep = random_nary(2000, "deep")
print("cây 'deep' 2000 node: h =", height_iter(big_deep))
for name, (ser, de) in CODECS.items():
    assert same_tree(de(ser(big_deep)), big_deep), name
assert run_codec_leetcode(Codec, to_leetcode_list(big_deep)) == to_leetcode_list(big_deep)
print("mọi codec (kể cả bản đệ quy, nhờ giới hạn 10^4) đúng trên cây sâu này")

300 cây ngẫu nhiên x 6 codec: round-trip đều đúng (cây cao nhất h = 146)
cây 'deep' 2000 node: h = 1003


mọi codec (kể cả bản đệ quy, nhờ giới hạn 10^4) đúng trên cây sâu này


## 13. So sánh độ dài chuỗi giữa các cách

Đếm theo **token** (đơn vị) trước, rồi đo **ký tự** thật trên cây ngẫu nhiên. Gọi `n` là số node, `I` là số node **có con** (node trong, internal), `n - I` là số lá:

| Cách | Token | Giải thích |
|---|---|---|
| Đếm con | đúng `2n` | mỗi node: `val` + `k` |
| Dấu `#` | đúng `2n` | mỗi node: `val` + `#` của chính nó |
| BFS `null` | `n + (n + 1) − (số null bị cắt)` ≤ `2n + 1` | mỗi node một nhóm kết thúc bằng `null`, cộng `null` sau root |
| Dấu ngoặc | `n` số + `2I` ngoặc (+ dấu cách giữa anh em) | lá không tốn ký hiệu |
| JSON list | `n` số + rất nhiều ngoặc và dấu phẩy | mỗi node `[v,[...]]`, lá còn thêm `[]` |

Về **ký tự**: token cấu trúc (`k`, `#`, `null`) có độ dài khác nhau — `k` thường chỉ 1 chữ số, `#` đúng 1 ký tự, còn `null` tới 4 ký tự. Cell dưới đo thật và `assert` các công thức token ở bảng trên.

In [27]:
def internal_count(root):
    return 0 if root is None else sum(1 for node in iter_nodes(root) if node.children)

def iter_nodes(root):
    stack = [root] if root else []
    while stack:
        node = stack.pop()
        yield node
        stack.extend(node.children or [])

random.seed(13)
print(f"{'cây':<14}{'n':>6}{'json':>8}{'count':>8}{'#':>8}{'ngoặc':>8}{'bfs':>8}")
for label, t in [("Example 2", root2), ("Example 1", root1),
                 ("wide 100", random_nary(100, "wide")), ("wide 1000", random_nary(1000, "wide")),
                 ("deep 1000", random_nary(1000, "deep")), ("sao 1000", build_nary(star))]:
    n = count_nodes(t)
    lens = {name: len(ser(t)) for name, (ser, _) in CODECS.items() if name != "count_iter"}
    print(f"{label:<14}{n:>6}{lens['json']:>8}{lens['count']:>8}{lens['sentinel']:>8}{lens['bracket']:>8}{lens['bfs']:>8}")
    # kiểm tra công thức token
    assert len(count_serialize(t).split(",")) == 2 * n
    assert len(sentinel_serialize(t).split(",")) == 2 * n
    assert len(bfs_serialize(t).split(",")) <= 2 * n + 1
    b = bracket_serialize(t)
    assert b.count("[") == b.count("]") == internal_count(t)
print("công thức số token ở bảng trên đều đúng")

cây                n    json   count       #   ngoặc     bfs
Example 2          6      39      23      23      13      21
Example 1         14      94      60      60      40      87
wide 100         100     945     694     694     543     964
wide 1000       1000    9392    6877    6876    5360    9861
deep 1000       1000    9188    6883    6883    5578    9873
sao 1000        1001    8895    5896    5893    3892    3896
công thức số token ở bảng trên đều đúng


### 13.2 Nhận xét từ bảng đo

- **Dấu ngoặc** ngắn nhất ở mọi dòng của bảng: lá chiếm phần lớn cây nhưng không tốn ký hiệu cấu trúc nào, chỉ node có con mới tốn một cặp `[` `]`.
- **Đếm con** và **`#`** đều đúng `2n` token và gần như dài bằng nhau: một số đếm nhỏ (thường 1 chữ số) và một dấu `#` đều tốn 1 ký tự. Chỉ ở node có ≥ 10 con (như root của cây "sao", số đếm `1000` tốn 4 ký tự) thì cách đếm con dài hơn vài ký tự.
- **JSON** và **BFS** là hai cách dài nhất trên cây ngẫu nhiên — và BFS thậm chí còn **dài hơn** JSON! Lý do: trong BFS, **mỗi node** (kể cả lá) gánh một nhóm kết thúc bằng `null,` — 5 ký tự — trong khi DFS chỉ tốn 1–2 ký tự cấu trúc cho mỗi node. JSON thì mỗi node tốn khoảng 5–6 ký tự ngoặc/phẩy.
- **Ngoại lệ thú vị — cây "sao":** BFS bỗng ngắn gần bằng dấu ngoặc. Toàn bộ 1000 lá nằm trong **một** nhóm, và 1000 `null` của các lá đều nằm ở **cuối** chuỗi nên bị cắt sạch. Việc "cắt `null` thừa ở cuối" giúp BFS rất nhiều khi lá dồn hết về tầng cuối, nhưng gần như không giúp gì khi lá rải rác ở nhiều tầng.

Kết luận thực dụng: độ dài các cách tự thiết kế đều là `O(n)` token và chênh nhau chỉ một hằng số nhỏ; khi phỏng vấn, hãy chọn cách **dễ chứng minh đúng và ít lỗi nhất** — cách 2 (đếm con) hoặc cách 3 (`#`).

## 14. Vì sao đúng? Độ phức tạp và so sánh các cách

### 14.1 Chứng minh cách 2 (đếm con) bằng quy nạp

Ký hiệu `S(T)` là dãy token serialize của cây `T`: nếu `T` có root giá trị `v` và các cây con `T1, ..., Tk` thì

`S(T) = v, k, S(T1), S(T2), ..., S(Tk)`.

**Mệnh đề:** nếu băng chuyền đang đứng ở đầu một đoạn `S(T)` (phía sau có thể còn token khác), thì một lời gọi `build()` đọc **đúng** các token của `S(T)` — không hơn, không kém — và trả về cây giống `T`.

Chứng minh bằng quy nạp theo số node của `T`:

- **Cơ sở** (`T` là lá, `S(T) = v, 0`): `build()` đọc `v` và `0`, vòng lặp chạy 0 lần, trả `Node(v)`. Đọc đúng 2 token. ✓
- **Bước quy nạp:** `build()` đọc `v, k`. Sau đó băng chuyền đứng ở đầu `S(T1)`; `T1` có ít node hơn `T` nên theo giả thiết quy nạp, lời gọi `build()` thứ nhất đọc đúng `S(T1)` và trả về `T1` — băng chuyền dừng ở đầu `S(T2)`. Lặp lại đến `Tk`. Tổng cộng đọc đúng `v, k, S(T1), ..., S(Tk) = S(T)`, và node trả về có giá trị `v`, các con `T1..Tk` đúng thứ tự. ✓

Áp dụng cho cả chuỗi: `deserialize(serialize(T))` đọc đúng toàn bộ chuỗi và trả về `T`. Hệ quả quan trọng: **vai trò của mỗi token (giá trị hay số con) được quyết định bởi vị trí đọc**, nên giá trị trùng với số con (như `1,3,3,2`) không gây nhầm lẫn. Tính chất "không đoạn `S(T)` nào là đoạn đầu của một `S(T')` khác" gọi là **prefix-free** — đó là lý do không cần dấu phân cách giữa các cây con.

### 14.2 Vì sao cách 3 và cách 4 đúng

- **Cách `#`:** `S(T) = v, S(T1), ..., S(Tk), #`. Quy nạp y hệt: `build(v)` lặp — mỗi token giá trị mở đầu một cây con (đọc trọn nhờ quy nạp), và token `#` đầu tiên gặp **ở cấp của nó** chính là `#` của `T` (các `#` bên trong `S(Ti)` đã bị lời gọi con "tiêu thụ").
- **Cách BFS:** bất biến (invariant) — "nhóm thứ `g` trong chuỗi là danh sách con của node thứ `g` theo thứ tự BFS". Serialize tạo ra bất biến này vì ghi nhóm theo thứ tự lấy node khỏi queue; deserialize dùng queue với **cùng** thứ tự (con được đưa vào queue đúng thứ tự xuất hiện), nên node thứ `g` lấy ra khỏi queue chính là chủ của nhóm `g`.

### 14.3 Độ phức tạp

`n` = số node, `h` = chiều cao, `w` = độ rộng lớn nhất (số node trên tầng đông nhất), `L` = độ dài chuỗi (`L = O(n)` vì mỗi giá trị ≤ 5 chữ số).

| Cách | Serialize | Deserialize | Độ dài chuỗi | Bộ nhớ phụ (ngoài cây và chuỗi) | Ghi chú |
|---|---|---|---|---|---|
| 1. JSON | `O(n)` | `O(n)` | dài (cùng nhóm với BFS) | `O(n)` object trung gian + `O(h)` đệ quy | "lười", phụ thuộc thư viện |
| **2. Đếm con (chính)** | `O(n)` | `O(n)` | `2n` token | `O(h)` call stack | dễ chứng minh nhất |
| 2'. Đếm con, bản lặp | `O(n)` | `O(n)` | `2n` token | `O(h)` stack tự quản lý | an toàn với cây rất sâu |
| 3. Dấu `#` | `O(n)` | `O(n)` | `2n` token | `O(h)` | code deserialize ngắn nhất |
| 3'. Dấu ngoặc | `O(L)` | `O(L)` | `n + 2I` token | `O(h)` stack | thường ngắn nhất |
| 4. BFS `null` | `O(n)` | `O(n)` | ≤ `2n + 1` token | `O(w)` queue | đúng định dạng LeetCode, không đệ quy |

Mọi cách đều phải chạm vào từng node ít nhất một lần, nên `O(n)` thời gian là **tối ưu**. Lưu ý `split(",")` tạo list `O(n)` token — đó là bộ nhớ cho dữ liệu đầu vào, không tính vào "bộ nhớ phụ" của thuật toán.

## 15. Lỗi thường gặp

### Lỗi 1: "Giấu" cây trong `self` (vi phạm stateless)

```python
class Codec:
    def serialize(self, root):
        self.saved = root          # SAI
        return "ok"
    def deserialize(self, data):
        return self.saved
```

Chạy với **cùng** một object thì có vẻ "pass", nhưng đề cấm lưu trạng thái, và chỉ cần serialize bằng object này, deserialize bằng object khác (hoặc ở máy khác!) là hỏng ngay. **Sửa:** mọi thông tin phải nằm trong chuỗi trả về.

### Lỗi 2: Chỉ ghi giá trị, không ghi cấu trúc

`"1,2,3"` ứng với nhiều cây (mục 5). **Sửa:** thêm số con, hoặc `#`, hoặc ngoặc, hoặc `null` ngăn nhóm.

### Lỗi 3: Không có dấu phân cách giữa các token

Nối thẳng `"".join(...)` ở cách đếm con: cây `112` (một node) cho `"112" + "0" = "1120"`, cây `1 → 2` cho `"1"+"1"+"2"+"0" = "1120"` — trùng nhau! **Sửa:** dùng `","` để nối.

### Lỗi 4: Cây rỗng và cái bẫy `"".split(",")`

`deserialize("")` mà không kiểm tra trước sẽ `int("")` → `ValueError`. **Sửa:** `if not data: return None` ở dòng đầu.

### Lỗi 5: Giả định `children` luôn là list

Nếu `children` là `None`, `for c in node.children` nổ `TypeError` và `len(node.children)` cũng vậy. **Sửa:** `node.children or []`.

### Lỗi 6: `tokens.pop(0)` trong deserialize

Chạy đúng nhưng mỗi lần `pop(0)` dịch cả list → `O(n²)`. **Sửa:** `iter` + `next`, chỉ số `i`, hoặc `deque.popleft()`.

### Lỗi 7: Quên `null` sau root ở cách BFS

Ghi `1,3,2,4,null,5,6` (thiếu `null` thứ hai) thì bộ đọc bỏ qua `3` (tưởng là `null`), và mọi nhóm phía sau bị lệch sang cha khác → cây sai mà **không báo lỗi**. **Sửa:** luôn ghi `root, null` rồi mới tới các nhóm.

### Lỗi 8: Đệ quy quá sâu và mutable default

Đệ quy trên cây `h = 1000` với giới hạn mặc định → `RecursionError` (mục 3.2); và `children=[]` làm mặc định → mọi node dùng chung một list (mục 2.2). **Sửa:** `sys.setrecursionlimit` hoặc bản lặp; luôn `Node(val, [])`.

Cell dưới tái hiện lần lượt các lỗi 1, 3, 4, 5, 6, 7.

In [28]:
# Lỗi 1: giấu cây trong self
class CodecCheat:
    def serialize(self, root):
        self.saved = root
        return "ok"
    def deserialize(self, data):
        return self.saved

c = CodecCheat()
print("Lỗi 1, cùng object  :", to_leetcode_list(c.deserialize(c.serialize(build_nary(EX2)))))
try:
    run_codec_leetcode(CodecCheat, EX2)
except AttributeError as e:
    print("Lỗi 1, hai object   :", type(e).__name__, "-", e)

# Lỗi 3: không có dấu phân cách
def count_serialize_nodelim(root):
    return count_serialize(root).replace(",", "")
print("Lỗi 3:", count_serialize_nodelim(Node(112)), "==", count_serialize_nodelim(Node(1, [Node(2)])))

# Lỗi 4: quên kiểm tra chuỗi rỗng
def bad_deserialize(data):
    it = iter(data.split(","))
    return Node(int(next(it)))
try:
    bad_deserialize(count_serialize(None))
except ValueError as e:
    print("Lỗi 4: ValueError -", e)

# Lỗi 5: children là None
leaf = Node(5); leaf.children = None
try:
    len(leaf.children)
except TypeError as e:
    print("Lỗi 5: TypeError -", e, "| sửa:", len(leaf.children or []))

# Lỗi 6: pop(0) so với iterator
toks = [str(i % 10) for i in range(40_000)]
t0 = time.perf_counter(); tmp = toks[:]
while tmp: tmp.pop(0)
t_pop = time.perf_counter() - t0
t0 = time.perf_counter(); it = iter(toks)
for _ in range(len(toks)): next(it)
t_it = time.perf_counter() - t0
print(f"Lỗi 6: 40 000 token: pop(0) {t_pop*1000:.1f} ms, iterator {t_it*1000:.1f} ms")

# Lỗi 7: quên null sau root
wrong = bfs_deserialize("1,3,2,4,null,5,6")
print("Lỗi 7: cây đọc ra =", to_leetcode_list(wrong), "(đúng phải là", EX2, ")")

Lỗi 1, cùng object  : [1, None, 3, 2, 4, None, 5, 6]
Lỗi 1, hai object   : AttributeError - 'CodecCheat' object has no attribute 'saved'
Lỗi 3: 1120 == 1120
Lỗi 4: ValueError - invalid literal for int() with base 10: ''
Lỗi 5: TypeError - object of type 'NoneType' has no len() | sửa: 0


Lỗi 6: 40 000 token: pop(0) 142.0 ms, iterator 4.5 ms
Lỗi 7: cây đọc ra = [1, None, 2, 4, None, 5, 6] (đúng phải là [1, None, 3, 2, 4, None, 5, 6] )


## 16. Mẹo nhận dạng pattern và bài liên quan

### Dấu hiệu nên nghĩ tới serialize / mã hóa cây

- Đề nói "encode / decode", "lưu và dựng lại", "chuyển cây thành chuỗi".
- Cần **so sánh / băm** (hash) cả một cây con: chuỗi serialize của cây con làm **khóa** (key) trong `dict` — hai cây con giống nhau ⇔ chuỗi giống nhau (vì định dạng là "một-một").
- Cần chuyển cây giữa hai cấu trúc khác nhau (N-ary ↔ nhị phân).

### Khung tư duy nhớ nhanh

> **Chuỗi = giá trị + thông tin cấu trúc.** Với mỗi node, hỏi: "người đọc làm sao biết danh sách con của node này **bắt đầu** và **kết thúc** ở đâu?" — trả lời bằng *số con* (báo trước), *`#`* (báo sau), *ngoặc* (cả hai) hoặc *nhóm theo tầng*.

### Bài liên quan

| Bài | Liên hệ |
|---|---|
| 297. Serialize and Deserialize Binary Tree (Hard) | anh em trực tiếp; cây nhị phân có đúng 2 chỗ con nên chỉ cần `#` cho chỗ trống |
| 449. Serialize and Deserialize BST (Medium) | tận dụng tính chất BST để bỏ hẳn thông tin cấu trúc |
| 331. Verify Preorder Serialization of a Binary Tree (Medium) | kiểm tra chuỗi hợp lệ mà không dựng cây — Bài 5 là phiên bản N-ary |
| 431. Encode N-ary Tree to Binary Tree (Hard, Premium) | mã hóa sang **cây** thay vì chuỗi — Bài 6 |
| 589 / 590 / 429. N-ary Tree Preorder / Postorder / Level Order Traversal | các phép duyệt nền tảng dùng ở mục 3 và 10 |
| 559. Maximum Depth of N-ary Tree | chính là `height` ở mục 3 |
| 1490. Clone N-ary Tree (Medium, Premium) | `deserialize(serialize(t))` là một cách clone (chậm hơn clone trực tiếp) |
| 652. Find Duplicate Subtrees (Medium) | serialize mỗi cây con làm khóa `dict` — demo phiên bản N-ary ngay dưới |

Demo: tìm các cây con **trùng nhau** trong cây N-ary bằng cách lấy chuỗi đếm-con của từng cây con làm khóa. (Cách này tốn `O(n·h)` vì mỗi cây con tạo lại chuỗi; bản tối ưu trong bài 652 gán mỗi chuỗi một **mã số** để khóa chỉ dài `O(k)`.)

In [29]:
def find_duplicate_subtrees_nary(root):
    seen, dups = {}, []

    def key(node):                         # post-order: khóa của con có trước
        kids = node.children or []
        k = ",".join([str(node.val), str(len(kids))] + [key(c) for c in kids])
        seen[k] = seen.get(k, 0) + 1
        if seen[k] == 2:                   # lần thứ 2 gặp -> ghi nhận đúng một lần
            dups.append(node)
        return k

    if root:
        key(root)
    return dups

t = build_nary([1, None, 2, 3, 2, None, 4, None, 5, None, 4])
print_tree(t)
print("cây con trùng (dạng LeetCode):", [to_leetcode_list(d) for d in find_duplicate_subtrees_nary(t)])

1
├── 2
│   └── 4
├── 3
│   └── 5
└── 2
    └── 4
cây con trùng (dạng LeetCode): [[4], [2, None, 4]]


## 17. Bài tập vận dụng

Làm theo thứ tự; tự làm **trước** rồi mới xem mục 18 (gợi ý) và mục 19 (đáp án).

### Bài 1 — Đoán output (không chạy code)

Cho cây:

```text
7
├── 0
│   └── 0
└── 10000
```

Viết ra (a) chuỗi của cách 2 (đếm con), (b) chuỗi của cách 3 (`#`), (c) chuỗi BFS kiểu LeetCode (đã cắt `null` cuối), (d) chuỗi dấu ngoặc. Mỗi chuỗi (a), (b) có bao nhiêu token?

### Bài 2 — Tính tay bằng bảng

(a) Vẽ cây của chuỗi đếm-con `"7,2,8,1,9,0,10,0"`. Lập bảng như mục 7.2 (lời gọi `build`, token đọc, quyết định).

(b) Vẽ cây của list LeetCode `[1,null,2,3,null,4,null,5,6]`. Lập bảng như mục 2.5 (parent lấy từ queue, nhóm con, queue sau bước).

(c) Chuỗi `"1,2,3,0"` có phải chuỗi đếm-con hợp lệ không? Nếu đưa vào `count_deserialize` thì chuyện gì xảy ra?

### Bài 3 — Điền TODO: viết lại lời giải chính

Hoàn thiện `CodecExercise` theo cách 2 (pre-order + số con). Yêu cầu: stateless, xử lý cây rỗng, dùng `iter` + `next`. Làm xong thì bỏ comment các dòng test.

In [30]:
class CodecExercise:
    def serialize(self, root: 'Node') -> str:
        # TODO 1: cây rỗng -> ""
        # TODO 2: tạo list out; viết dfs(node): ghi str(val), ghi str(số con), rồi dfs từng con
        # TODO 3: trả ",".join(out)
        pass

    def deserialize(self, data: str) -> 'Node':
        # TODO 4: chuỗi rỗng -> None (kiểm tra TRƯỚC khi split)
        # TODO 5: it = iter(data.split(","))
        # TODO 6: viết build(): đọc val, đọc k, tạo Node(val, []), gọi build() k lần để gắn con
        # TODO 7: trả build()
        raise NotImplementedError


# Tự làm xong rồi mới mở test:
# for vals in [EX1, EX2, EX3, [7], star]:
#     assert run_codec_leetcode(CodecExercise, vals) == vals
# print("Bài 3 PASS")

### Bài 4 — Tự viết lại: deserialize định dạng `#` **không đệ quy**

Viết `sentinel_deserialize_iter(data)` đọc chuỗi của cách 3 (`"1,3,5,#,6,#,#,2,#,4,#,#"`) bằng một **stack** các node "đang mở", không dùng đệ quy. Phải chạy được trên cây xích `10^4` tầng.

In [31]:
def sentinel_deserialize_iter(data):
    # TODO 1: chuỗi rỗng -> None
    # TODO 2: tách token; token đầu là root; stack = [root]
    # TODO 3: duyệt các token còn lại:
    #         - "#": node trên đỉnh stack đã hết con -> ...
    #         - giá trị: tạo node mới, gắn vào ..., rồi ...
    # TODO 4: trả root
    raise NotImplementedError


# Tự làm xong rồi mới mở test:
# assert same_tree(sentinel_deserialize_iter("1,3,5,#,6,#,#,2,#,4,#,#"), root2)
# assert sentinel_deserialize_iter("") is None
# chain_str = ",".join([str(v) for v in range(10_000)] + ["#"] * 10_000)   # cây xích 10^4 tầng
# assert same_tree(sentinel_deserialize_iter(chain_str), make_chain(10_000))
# print("Bài 4 PASS")

### Bài 5 — Biến thể: kiểm tra chuỗi đếm-con hợp lệ mà không dựng cây (kiểu LeetCode 331)

Viết `is_valid_count_serialization(data) -> bool`: trả `True` khi và chỉ khi `data` là chuỗi serialize (cách 2) của **một** cây N-ary nào đó. Quy ước `""` (cây rỗng) là hợp lệ. Không được dựng cây; mục tiêu `O(n)` thời gian, `O(1)` bộ nhớ phụ (không tính list token).

Ví dụ: `"1,3,3,2,5,0,6,0,2,0,4,0"` → `True`; `"1,2,3,0"` → `False` (thiếu một con); `"1,0,2,0"` → `False` (thừa token sau khi cây đã xong); `"1,1"` → `False`; `"5"` → `False` (số token lẻ).

In [32]:
def is_valid_count_serialization(data: str) -> bool:
    # TODO 1: "" -> True; số token lẻ -> False
    # TODO 2: slots = 1  (số "chỗ trống" đang chờ một cây con; ban đầu chờ root)
    # TODO 3: duyệt từng cặp (val, k): nếu slots == 0 -> False (thừa); slots -= 1; slots += k
    # TODO 4: cuối cùng phải ... ?
    pass


# Tự làm xong rồi mới mở test:
# assert is_valid_count_serialization("1,3,3,2,5,0,6,0,2,0,4,0")
# assert not is_valid_count_serialization("1,2,3,0")
# assert not is_valid_count_serialization("1,0,2,0")
# assert not is_valid_count_serialization("1,1")
# assert not is_valid_count_serialization("5")
# assert is_valid_count_serialization("")
# print("Bài 5 PASS")

### Bài 6 — Mini challenge: LeetCode 431. Encode N-ary Tree to Binary Tree (Hard, Premium)

Thay vì mã hóa thành **chuỗi**, hãy mã hóa cây N-ary thành một **cây nhị phân** (`TreeNode` có `left`, `right`) sao cho giải mã lại được đúng cây ban đầu. Signature trên LeetCode:

```python
class Codec:
    def encode(self, root: 'Optional[Node]') -> Optional[TreeNode]:
    def decode(self, data: Optional[TreeNode]) -> 'Optional[Node]':
```

Trong notebook đặt tên class là `Codec431` để không đè `Codec` ở trên. Gợi ý lớn: biểu diễn **"con đầu – anh kế"** (left-child right-sibling).

In [33]:
class TreeNode:
    def __init__(self, x):
        self.val = x
        self.left = None
        self.right = None


class Codec431:
    def encode(self, root):
        # TODO 1: None -> None
        # TODO 2: tạo TreeNode(root.val)
        # TODO 3: con ĐẦU TIÊN của root -> đặt ở .left (đã encode)
        # TODO 4: các con tiếp theo -> nối thành chuỗi qua .right của con trước đó
        pass

    def decode(self, data):
        # TODO 5: None -> None
        # TODO 6: tạo Node(data.val, []); đi từ data.left theo .right, decode từng node, gắn làm con
        pass


# Tự làm xong rồi mới mở test:
# c = Codec431()
# for vals in [EX1, EX2, EX3, star]:
#     assert to_leetcode_list(c.decode(c.encode(build_nary(vals)))) == vals
# print("Bài 6 PASS")

## 18. Gợi ý theo mức

**Bài 1.** (1) Làm theo pre-order `7, 0, 0, 10000`. (2) Đếm con: mỗi node ghi `val, k`; `#`: mỗi node ghi `val`, các con, rồi `#`. (3) BFS: `7, null`, rồi nhóm con của `7`, của `0` (tầng 1), của `10000`, của `0` (tầng 2)... rồi cắt `null` cuối.

**Bài 2.** (1) Mỗi lời gọi `build` đọc đúng 2 token rồi gọi `k` lần chính nó. (2) `7` cần 2 con; con thứ nhất `8` cần 1 con... (3) Với (c): đếm xem cần bao nhiêu cây con và chuỗi cung cấp bao nhiêu; `next` trên băng chuyền đã hết sẽ ném gì?

**Bài 3.** (1) Chép tinh thần mục 6, nhưng tự gõ. (2) `build` lồng trong `deserialize` để dùng chung `it`. (3) `for _ in range(k): node.children.append(build())`.

**Bài 4.** (1) Stack chứa các node mà `#` của chúng **chưa** xuất hiện. (2) Gặp giá trị: con mới gắn vào `stack[-1]`, rồi chính nó cũng "đang mở". (3) Gặp `#`: `stack.pop()`.

**Bài 5.** (1) Mỗi cặp `(val, k)` **lấp** một chỗ trống và **mở** `k` chỗ mới. (2) Nếu chỗ trống về 0 mà vẫn còn token → thừa. (3) Cuối cùng `slots` phải đúng bằng `0` (không còn con nào đang chờ).

**Bài 6.** (1) Node N-ary `u` có con `c1, c2, ..., ck` ↦ trong cây nhị phân: `u.left = c1`, `c1.right = c2`, `c2.right = c3`, ... (2) Encode đệ quy cho từng con, nối chúng qua `.right`. (3) Decode: `cur = data.left; while cur: node.children.append(decode(cur)); cur = cur.right`.

## 19. Đáp án tham khảo

### Đáp án Bài 1

(a) Đếm con: `7,2,0,1,0,0,10000,0` — 8 token (`2n` với `n = 4`).
(b) Dấu `#`: `7,0,0,#,#,10000,#,#` — 8 token.
(c) BFS: `7,null,0,10000,null,0` — `7, null`; con của `7`: `0, 10000, null`; con của `0` (tầng 1): `0, null`; con của `10000`: `null`; con của `0` (tầng 2): `null` → cắt 3 `null` cuối.
(d) Dấu ngoặc: `7[0[0] 10000]`.

### Đáp án Bài 2

(a) `"7,2,8,1,9,0,10,0"`: `build#1` đọc `7, 2` → cần 2 con; `build#2` đọc `8, 1` → cần 1 con; `build#3` đọc `9, 0` → lá, gắn vào `8`; `8` đủ con → gắn vào `7`; `build#4` đọc `10, 0` → lá, gắn vào `7`. Cây:

```text
7
├── 8
│   └── 9
└── 10
```

(b) `[1,null,2,3,null,4,null,5,6]`: root `1`, bỏ `null`; parent `1` nhận `2, 3` (queue `[2, 3]`); parent `2` nhận `4` (queue `[3, 4]`); parent `3` nhận `5, 6` (hết list). Cây:

```text
1
├── 2
│   └── 4
└── 3
    ├── 5
    └── 6
```

(c) Không hợp lệ: `1` báo có 2 con, nhưng sau cặp `3, 0` (con thứ nhất) thì hết token. `count_deserialize` gọi `next(it)` trên băng chuyền rỗng → ném `StopIteration`. (Muốn báo lỗi rõ ràng hơn, có thể bắt nó và ném `ValueError("chuỗi không hợp lệ")`; hoặc kiểm tra trước bằng hàm của Bài 5.) Cell dưới kiểm chứng tất cả.

In [34]:
# Kiểm chứng Bài 1 và Bài 2
ex_tree = Node(7, [Node(0, [Node(0)]), Node(10000)])
assert count_serialize(ex_tree) == "7,2,0,1,0,0,10000,0"
assert sentinel_serialize(ex_tree) == "7,0,0,#,#,10000,#,#"
assert bfs_serialize(ex_tree) == "7,null,0,10000,null,0"
assert bracket_serialize(ex_tree) == "7[0[0] 10000]"
print("Bài 1 đúng")

t2a = count_deserialize("7,2,8,1,9,0,10,0")
assert same_tree(t2a, Node(7, [Node(8, [Node(9)]), Node(10)]))
t2b = build_nary([1, None, 2, 3, None, 4, None, 5, 6])
assert same_tree(t2b, Node(1, [Node(2, [Node(4)]), Node(3, [Node(5), Node(6)])]))
try:
    count_deserialize("1,2,3,0")
    print("không lỗi?!")
except StopIteration:
    print("Bài 2c: '1,2,3,0' -> StopIteration (thiếu con)")
print("Bài 2 đúng")

Bài 1 đúng
Bài 2c: '1,2,3,0' -> StopIteration (thiếu con)
Bài 2 đúng


In [35]:
# ---------- Đáp án Bài 3
class CodecExerciseSolution:
    def serialize(self, root: 'Node') -> str:
        if root is None:
            return ""
        out = []

        def dfs(node):
            kids = node.children or []
            out.append(str(node.val))
            out.append(str(len(kids)))
            for c in kids:
                dfs(c)

        dfs(root)
        return ",".join(out)

    def deserialize(self, data: str) -> 'Node':
        if not data:
            return None
        it = iter(data.split(","))

        def build():
            val, k = int(next(it)), int(next(it))
            node = Node(val, [])
            for _ in range(k):
                node.children.append(build())
            return node

        return build()

for vals in [EX1, EX2, EX3, [7], star]:
    assert run_codec_leetcode(CodecExerciseSolution, vals) == vals
print("Bài 3 PASS")


# ---------- Đáp án Bài 4
def sentinel_deserialize_iter_solution(data):
    if not data:
        return None
    toks = data.split(",")
    root = Node(int(toks[0]), [])
    stack = [root]                          # các node mà '#' của chúng chưa xuất hiện
    for tok in toks[1:]:
        if tok == "#":
            stack.pop()                     # node trên đỉnh đã hết con
        else:
            child = Node(int(tok), [])
            stack[-1].children.append(child)
            stack.append(child)             # con mới cũng đang "mở"
    return root

assert same_tree(sentinel_deserialize_iter_solution("1,3,5,#,6,#,#,2,#,4,#,#"), root2)
assert sentinel_deserialize_iter_solution("") is None
chain_str = ",".join([str(v) for v in range(10_000)] + ["#"] * 10_000)
assert same_tree(sentinel_deserialize_iter_solution(chain_str), make_chain(10_000))
random.seed(4)
for _ in range(100):
    t = random_nary(random.randint(0, 60), random.choice(["wide", "deep"]), 0, 5)
    assert same_tree(sentinel_deserialize_iter_solution(sentinel_serialize(t)), t)
print("Bài 4 PASS")


# ---------- Đáp án Bài 5
def is_valid_count_serialization_solution(data: str) -> bool:
    if not data:
        return True
    toks = data.split(",")
    if len(toks) % 2:
        return False
    slots = 1                               # chờ root
    for j in range(0, len(toks), 2):
        if slots == 0:                      # cây đã xong mà còn token
            return False
        slots -= 1                          # cặp này lấp một chỗ
        slots += int(toks[j + 1])           # và mở k chỗ cho con
    return slots == 0

assert is_valid_count_serialization_solution("1,3,3,2,5,0,6,0,2,0,4,0")
assert not is_valid_count_serialization_solution("1,2,3,0")
assert not is_valid_count_serialization_solution("1,0,2,0")
assert not is_valid_count_serialization_solution("1,1")
assert not is_valid_count_serialization_solution("5")
assert is_valid_count_serialization_solution("")
random.seed(5)
for _ in range(200):                        # đối chiếu với việc thử dựng cây thật
    t = random_nary(random.randint(1, 8), "wide", 0, 3)
    toks = count_serialize(t).split(",")
    if random.random() < 0.5:               # làm hỏng ngẫu nhiên một số đếm
        j = 2 * random.randrange(len(toks) // 2) + 1
        toks[j] = str(random.randint(0, 3))
    s = ",".join(toks)
    try:
        it_ok = count_deserialize(s) is not None and count_serialize(count_deserialize(s)) == s
    except StopIteration:
        it_ok = False
    assert is_valid_count_serialization_solution(s) == it_ok, s
print("Bài 5 PASS")


# ---------- Đáp án Bài 6 (LeetCode 431, left-child right-sibling)
class Codec431Solution:
    def encode(self, root):
        if root is None:
            return None
        b = TreeNode(root.val)
        prev = None
        for child in root.children or []:
            enc = self.encode(child)
            if prev is None:
                b.left = enc                # con đầu -> left
            else:
                prev.right = enc            # các con sau -> nối qua right
            prev = enc
        return b

    def decode(self, data):
        if data is None:
            return None
        node = Node(data.val, [])
        cur = data.left
        while cur:
            node.children.append(self.decode(cur))
            cur = cur.right
        return node

c431 = Codec431Solution()
for vals in [EX1, EX2, EX3, star]:
    assert to_leetcode_list(c431.decode(c431.encode(build_nary(vals)))) == vals
b = c431.encode(root2)
print("Ex2 -> cây nhị phân: root", b.val, "| left", b.left.val, "| left.right", b.left.right.val,
      "| left.left", b.left.left.val, "| left.left.right", b.left.left.right.val)
print("Bài 6 PASS")

Bài 3 PASS
Bài 4 PASS
Bài 5 PASS
Ex2 -> cây nhị phân: root 1 | left 3 | left.right 2 | left.left 5 | left.left.right 6
Bài 6 PASS


## 20. Template cuối cùng nên nhớ

```python
class Codec:
    def serialize(self, root):
        if root is None:
            return ""
        out = []
        def dfs(node):
            kids = node.children or []
            out.append(str(node.val))
            out.append(str(len(kids)))
            for c in kids:
                dfs(c)
        dfs(root)
        return ",".join(out)

    def deserialize(self, data):
        if not data:
            return None
        it = iter(data.split(","))
        def build():
            node = Node(int(next(it)), [])
            for _ in range(int(next(it))):
                node.children.append(build())
            return node
        return build()
```

> *Lưu ý nhỏ khi tự gõ lại:* trong hàm lồng `dfs`, đừng viết `out += [...]` — phép gán `+=` khiến Python coi `out` là biến cục bộ của `dfs` và báo `UnboundLocalError`. Dùng `out.append(...)` như trên là an toàn. Nhớ thêm `sys.setrecursionlimit(10**4)` ở đầu file.

Câu thần chú: **"Pre-order: ghi mình, ghi số con, rồi tới các con. Đọc lại: đọc mình, đọc số con, dựng đúng chừng ấy cây con."**

## 21. Checklist tự kiểm tra trước khi submit

- [ ] Class tên `Codec`, hai method `serialize(self, root)` và `deserialize(self, data)` đúng chữ ký.
- [ ] **Stateless**: không lưu gì vào `self`, không dùng biến global.
- [ ] Cây rỗng: `serialize(None)` trả chuỗi hợp lệ; `deserialize` kiểm tra chuỗi rỗng **trước** khi `split`.
- [ ] Có **dấu phân cách** giữa các token (giá trị nhiều chữ số).
- [ ] Ghi đủ thông tin cấu trúc (số con / `#` / ngoặc / `null`) — không chỉ ghi giá trị.
- [ ] Giữ đúng **thứ tự** các con.
- [ ] Phòng thủ `node.children or []`; tạo node mới bằng `Node(val, [])`.
- [ ] Đọc token bằng `iter` + `next` hoặc chỉ số — không `pop(0)`.
- [ ] Cây sâu `h = 1000`: đã `sys.setrecursionlimit(10**4)` hoặc dùng bản lặp / BFS.
- [ ] Đã test round-trip trên 3 ví dụ của đề, cây một node, cây sao, cây xích, và cây ngẫu nhiên.